# ST -673 
## Machine Learning Project
**Team:**
* Shaikh Naima Labiba (s3954789)
* Daniel Rowley
* GeonHyeong Do
* Julian Mariano 


## Notebook Skeleton  TODO: organize the notebook


### 1. Problem Statement and Clinical Goal
- Predict referral category: Cat 1, Cat 2, Cat 3
- Priority: maximize urgent-case safety (Cat 1 recall)

### 2. Data Loading and Quality Checks
- Load CSV
- Parse dates
- Check missing values, duplicates, invalid dates

### 3. Exploratory Data Analysis
- Class distribution (Cat 1/2/3)
- Waiting-time distribution and breach rates
- Key feature relationships and correlations

### 4. Feature Engineering and Leakage Control
- Keep only referral-time features
- Drop post-procedure leakage variables
- Define numeric and categorical columns

### 5. Train/Test Split and Preprocessing
- Stratified split
- Build preprocessing pipeline (imputation, scaling, one-hot encoding)

### 6. Baseline Models
- Train baseline set (Dummy, Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, SVM, KNN)
- Compare with Cat 1 recall, macro F1, balanced accuracy
- Confusion matrix for best baseline

### 7. Enhanced and Tuned Models
- Tune Logistic Regression and Random Forest
- Tune XGBoost and LightGBM
- Fit Extra Trees benchmark

### 8. Unified Model Comparison
- Use a single comparison table for all candidate models
- Rank by macro F2, then Cat 1 recall, then macro F1

### 9. Explainability and Clinical Interpretation
- Feature importance / coefficients
- Symptom-related contributors
- Operational implications for urgent-case triage

### 10. Final Recommendation
- Final selected model
- Why it was selected
- Risks, limitations, and next steps

### Imports

In [ ]:
import os
from IPython.display import display
import warnings

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

### Load Data

In [ ]:
df = pd.read_csv('endoscopy_patients.csv')

# formatting the date variables
df['referral_date'] = pd.to_datetime(df['referral_date'], dayfirst=True)
df['procedure_date'] = pd.to_datetime(df['procedure_date'], dayfirst=True)

## validating the load
## TODO: remove one of the redundant description calls idk which one ill pick yet
print(df.shape) 
print(df.info)
print(df.describe)
pd.set_option('display.max_column', None)  # as i want to inspect all the columns
df.head()

In [ ]:
df.isnull().sum()
df.duplicated().sum()
df.describe()

In [ ]:
(df['procedure_date'] < df['referral_date']).sum()

### Exploring Complication Type

In [ ]:
prod = df['complication_type'].dropna()
print(f"\nNone Empty Complication Types ({len(prod)} / {len(df['complication_type'])}):\n")

for type in prod.unique():
    print(type)

As we can see above, since complication type appears so infrequently in the dataset we have regarded it as statistically insignificant and not useful to account for in our predictive model.

### Analysing BMI

In [ ]:
# Get BMI Data
bmi_prod = df['bmi']

# Min/Max scale data
bmi_prod = (bmi_prod - min(bmi_prod)) / (max(bmi_prod) - min(bmi_prod))

# Plot distribution
plt.hist(bmi_prod)
plt.show()

As we can see above, after minmax scaling the BMI distribution could be considered normally distriubted. With relatively even tails and a centered median and mean.

In [ ]:
# Get risk score Data
risk_prod = df['risk_score']

# Min/Max scale data
risk_prod = (risk_prod - min(risk_prod)) / (max(risk_prod) - min(risk_prod))

# Plot distribution
plt.hist(risk_prod)
plt.show()

We can observe a less normally distributed arrangement here with the normalised risk score.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.regplot(
    x='bmi',
    y='risk_score',
    data=df,
    scatter_kws={'alpha': 0.3},  # transparency helps with overplotting
    line_kws={'color': 'red'},
    ax=ax
)

ax.set_title('BMI vs Risk Score with Regression Line')
ax.set_xlabel('BMI')
ax.set_ylabel('Risk Score')
plt.show()

The regression line shows no meaningful relationship, confirming the lack of correlation between BMI and risk score.

In [ ]:
print(df[['bmi', 'risk_score']].describe())
print("\nPearson correlation:", df['bmi'].corr(df['risk_score']))
print("Spearman correlation:", df['bmi'].corr(df['risk_score'], method='spearman'))

Since both are near 0, that means there is no meaningful linear (pearson) or monotonic (spearman) relationship between BMI and risk score :(

Therefore, BMI at least on its own is not a reliable predictor of risk score.

In [ ]:
bins = [0, 18.5, 25, 30, 35, 40, float('inf')]
labels = ['Underweight', 'Normal', 'Overweight', 'Obese I', 'Obese II', 'Obese III']

# Puts stuff into each category based on bins
df['bmi_category'] = pd.cut(df['bmi'], bins=bins, labels=labels)

# Box plot of risk score per BMI category
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(x='bmi_category', y='risk_score', data=df, ax=ax)
ax.set_title('Risk Score Distribution by BMI Category')
ax.set_xlabel('BMI Category')
ax.set_ylabel('Risk Score')
plt.show()

print(df.groupby('bmi_category')['risk_score'].agg(['mean', 'median', 'count']))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.scatterplot(
    x='bmi',
    y='risk_score',
    hue='gender',        
    data=df,
    alpha=0.4,
    ax=ax
)

# Add separate regression lines per gender (not very helpful here I know)
for g, group_data in df.groupby('gender'):
    sns.regplot(
        x='bmi', y='risk_score',
        data=group_data,
        scatter=False, 
        ax=ax,
        label=f'Gender {g} trend'
    )

ax.set_title('BMI vs Risk Score by Gender')
plt.show()

Gender doesn't seem to reveal any underlying nature.

In [ ]:
# Finding outliers
Q1 = df['bmi'].quantile(0.25)
Q3 = df['bmi'].quantile(0.75)
IQR = Q3 - Q1   # Inter-quartile range

outliers = df[(df['bmi'] < Q1 - 1.5 * IQR) | (df['bmi'] > Q3 + 1.5 * IQR)]
print(f"BMI outliers: {len(outliers)} rows")
print(outliers[['patient_id', 'bmi', 'risk_score']])

As we can see above, very few outliers which is great for our purposes.

In [ ]:
numeric_cols = df.select_dtypes(include='number').drop(columns=['patient_id'], errors='ignore')
mask = numeric_cols.corr().abs() < 0.25     # Creates a cutoff for weak correlations (below 0.25 mag)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    numeric_cols.corr(),
    mask=mask,
    annot=False,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    ax=ax
)
ax.set_title('Correlation Matrix - All Numeric Features')
plt.tight_layout()
plt.show()

We can see here again that the correlation between BMI and risk score has a magnitude below 0.25 (weak correlation).

Few things to note here:
 - cat_referral strongly negatively correlated with risk score
 - nbcsp strongly positively correlated with risk score
 - ind_fobt strong pos with nbcsp
 - strong pos corr between cat_referral and recommended max wait days (which is related to risk score)
 - strong neg corr between cat_referral and follow_up_months

### Create the target variable

In [ ]:
# TODO: Decide on the thresholds after we inspect the distribution
# TODO: create 3 cat BINS
# TODO: check class imbalance
# TODO: check data quality
# TODO: analyze the target variable
# TODO: map out the correlation between the risk categories vs maybe demographics?
# TODO: Analyze the waiting time
# TODO: add some benchmarks like MSR by each flag? or msybe prevalence rate
# TODO: create a correlation heatmap
# TODO: SUMMARY

### Check Data QUality

In [ ]:
# TODO: missing values
# TODO: Duplicates
# TODO: add more checks

### Demographics

In [ ]:
# 1. Age Analysis

print(df['age_yrs'].describe())
print(f"Median: {df['age_yrs'].median()}")
print(f"Skewness: {df['age_yrs'].skew():.2f}")

In [ ]:
# Age Distribution + Boxplot

fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Histogram
sns.histplot(df['age_yrs'], bins=20, ax=axes[0])
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age (Years)")
axes[0].set_ylabel("Frequency")

# Boxplot
sns.boxplot(x=df['age_yrs'], ax=axes[1])
axes[1].set_title("Age Boxplot")
axes[1].set_xlabel("Age (Years)")

plt.tight_layout()
plt.show()

# Stats
print(df['age_yrs'].describe())

The age distribution shows that most patients fall between 50 and 70 years, with a mean age of approximately 60. This indicates that the dataset is dominated by older individuals, which aligns with clinical expectations for colorectal conditions. A histogram was used to display the distribution of age, allowing patterns such as spread and concentration to be identified. A boxplot was included to highlight outliers and summarise the distribution using quartiles and median.

In [ ]:
df['age_group'] = pd.cut(
    df['age_yrs'],
    bins=[0, 40, 50, 60, 70, 100],
    labels=['<40', '40-50', '50-60', '60-70', '70+']
)

sns.countplot(x='age_group', data=df)
plt.title("Age Group Distribution")
plt.xlabel("Age Group")
plt.ylabel("Count")
plt.show()

Just added this because it makes the age categorisation cleaner and simpler with respect to clinically meaningful age ranges.

In [ ]:
# Gender Distribution

gender_labels = df['gender'].map({0: 'Female', 1: 'Male'})

fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Bar chart
sns.countplot(x=gender_labels, ax=axes[0])
axes[0].set_title("Distribution of Patients by Gender")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Count")

# Pie chart
gender_labels.value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    ax=axes[1],
    startangle=90
)
axes[1].set_title("Proportion of Patients by Gender")
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# Stats
print(gender_labels.value_counts())
print(gender_labels.value_counts(normalize=True))

The dataset is evenly distributed between male and female patients, with each group representing approximately 50% of the total population. This balance reduces potential gender bias in further analysis. A bar chart was used to show the frequency of each gender, making comparisons straightforward. A pie chart was included to clearly represent the proportion of each group within the dataset.

In [ ]:
# ASA Score Distribution

fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Bar chart
sns.countplot(x='asa_score', data=df, order=sorted(df['asa_score'].unique()), ax=axes[0])
axes[0].set_title("Distribution of ASA Scores")
axes[0].set_xlabel("ASA Score")
axes[0].set_ylabel("Count")

# Pie chart
df['asa_score'].value_counts().sort_index().plot(
    kind='pie',
    autopct='%1.1f%%',
    ax=axes[1],
    startangle=90
)
axes[1].set_title("Proportion of ASA Scores")
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# Stats
print(df['asa_score'].value_counts())
print(df['asa_score'].value_counts(normalize=True))

Most patients fall within ASA scores of 2 and 3, indicating moderate health risk. A smaller proportion of patients are classified as ASA 1 and ASA 4, representing lower and higher-risk groups respectively. A bar chart was used to compare the number of patients across ASA categories. A pie chart was included to visualise the proportional distribution of health risk levels.

### Waiting Time

In [ ]:

# 1. how skewed is the wait time distribution?

print(df['waiting_days'].describe())
print(f"\nMedian: {df['waiting_days'].median()}")
print(f"Skewness: {df['waiting_days'].skew():.2f}")

 INSIGHT: Right-skewed (1.84) - median (46) is more representative than mean (52).
 Non-parametric methods preferred. Log-transform worth exploring for linear models.

In [ ]:
# 2. Histogram + Distriribution
## What shape is the distribution? Any multimodality?

fig, axes = plt.subplots(1, 2, figsize=(14,5))

axes[0].hist(df['waiting_days'], bins = 30, edgecolor = 'black', color ='#E0218A')
axes[0].set_title('Distribution of Waiting Time')
axes[0].set_xlabel('Waiting Days')
axes[0].set_ylabel('Frequency')

df['waiting_days'].plot(kind = 'kde', ax=axes[1], color = 'lightpink')
axes[1].set_title('KDE - Waiting Days')
axes[1].set_xlabel('Waiting Days')

plt.tight_layout()
plt.show()



INSIGHT: Heavy right tail with a strong left peak - likely the three referral

categories overlaid. Stratifying by cat_referral will be essential.

In [ ]:
# 3. Boxplot

# Where do most patients sit, and how extreme are the outliers?

plt.figure(figsize=(8,4))
plt.boxplot(df['waiting_days'].dropna(), vert = False, patch_artist= True, boxprops = dict(facecolor = 'lightpink', color = 'black'))

plt.title('Boxplot of Waiting Days')
plt.xlabel('Waiting Days')
plt.show()

INSIGHT: IQR 18 to 71 days. Long right whisker = a subset with severely delayed care, almost certainly Cat 3 referrals.

In [ ]:
# 4. Outliers (beyond 1.5x IQR)
# How many patients have clinically extreme wait times?


Q1 = (df['waiting_days'].quantile(0.25))
Q3 = (df['waiting_days'].quantile(0.75))
IQR = Q3 - Q1

outliers = df[df['waiting_days'] > Q3 + 1.5 * IQR]
print(f"\nOutliers (>(Q3 + 1.5 * IQR : .0f days): {len(outliers)} patients")
print(outliers['waiting_days'].describe())

INSIGHT: 72 patients, mean 198 days, max 296 - real capacity problem,
not data errors. Almost certainly Cat 3.

In [ ]:
# 5. Waiting Days by Referral Category
# Does referral category cleanly separate waiting times?


# Mapping
cat_labels = {1: 'Urgent(Cat 1)', 2: 'Semi-Urgent(Cat 2)', 3: 'Routine: Cat 3'}
df['cat_referral_label'] = df['cat_referral'].map(cat_labels)

# Summary stats per catergory

print(df.groupby('cat_referral_label')['waiting_days'].describe())


INSIGHT: Cat 1 near-uniform. Cat 2 has the widest IQR relative to its median - capacity pressure showing.

In [ ]:
# 6. Boxplot of Category
# How does spread differ across categories?

plt.figure(figsize = (8,5))
sns.boxplot(data = df, x ='cat_referral_label', y = 'waiting_days', hue='cat_referral_label', palette='magma', legend= False, order = ['Urgent(Cat 1)', 'Semi-Urgent(Cat 2)', 'Routine (Cat 3)'] )
plt.title('Waiting Days by Referral  Category')
plt.xlabel('Referral_Category')
plt.ylabel('Waiting Days')
plt.show()



INSIGHT: 4.8% overall, but Cat 2 hits 7.6% - concerning given it's the largest group (n=961). Cat 3's 0% is a threshold artefact, not efficiency.

In [ ]:
# 7. Median per Category
print("\nMedian waiting days per category:")
print(df.groupby('cat_referral_label')['waiting_days'].median().sort_values())

In [ ]:
# 8. Waiting Days aagainst Recommended Max Wait

# Create Breach column
df['exceeded_wait'] = df['waiting_days'] > df['recommended_max_wait_days']

#Overall breach rate

total = len(df)
breached = df['exceeded_wait'].sum()
print(f"Patients who exceeded recommended wait: {breached} / {total} ({breached / total * 100 : .1f} % )")


# Breach rate per referral category

breach_by_cat = df.groupby('cat_referral_label')['exceeded_wait'].mean() * 100
print("\nBreach rate per category (%): ")
print(breach_by_cat.sort_index())

In [ ]:
 # 9. Bar chart: breached vs not breached per category
 # Which category drives the most breaches in absolute terms?

breach_counts = df.groupby(['cat_referral_label', 'exceeded_wait']).size().unstack()
breach_counts.columns = ['Within Limit', 'Exceeded']
breach_counts = breach_counts.loc[['Urgent(Cat 1)', 'Semi-Urgent(Cat 2)', 'Routine: Cat 3']]

breach_counts.plot(kind='bar', figsize=(9, 5), color=['lightpink', 'purple'], edgecolor='black')
plt.title('Wait Time: Within vs Exceeded Recommended Limit by Category')
plt.xlabel('Referral Category')
plt.ylabel('Number of Patients')
plt.xticks(rotation=15)
plt.legend(title='Status')
plt.tight_layout()
plt.show()


INSIGHT: Cat 2 dominates breach counts due to sheer volume. The priority tier for any triage model.

In [ ]:
# 10. Waiting Days over time
# Has average wait time changed over the 2010-2024 study period?
# Extract year-month from referral date
df['referral_yearmonth'] = df['referral_date'].dt.to_period('M')

# Average waiting days per month
monthly_avg = df.groupby('referral_yearmonth')['waiting_days'].mean()

In [ ]:
# Plot
plt.figure(figsize=(14, 5))
monthly_avg.plot(color='lightpink')
plt.title('Average Waiting Days by Referral Month')
plt.xlabel('Referral Month')
plt.ylabel('Average Waiting Days')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# check by year 
df['referral_year'] = df['referral_date'].dt.year
print("\nAverage waiting days per year:")
print(df.groupby('referral_year')['waiting_days'].mean().round(1))



INSIGHT: Rolling average confirms long-run stability. Log-transform gives near-symmetric distribution - useful for MLP/linear, not needed for trees.

In [ ]:
# Do peak referral months (Jun/Jul/Aug/Dec) inflate waiting times?
df['month'] = df['referral_date'].dt.month
df['is_peak'] = df['month'].isin([6,7,8,12])
sns.boxplot(data=df, x='is_peak', y='waiting_days', color="#E0218A")
plt.title('Waiting Days: Peak vs Non-Peak Referral Months')
plt.show()

INSIGHT: Minimal difference. Category system buffers seasonal demand, is_peak is likely low-value as a feature.

In [ ]:
 # 10. Waiting Days vs Other Variables
 # Do procedure type, gender, age, or ASA score independently predict wait time?


# ny Procedure Type
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='procedure_type', y='waiting_days',  hue='procedure_type', palette='magma', legend= False)
plt.title('Waiting Days by Procedure Type')
plt.xlabel('Procedure Type')
plt.ylabel('Waiting Days')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

  #  By Gender 
plt.figure(figsize=(6, 5))
sns.boxplot(data=df, x='gender', y='waiting_days',  hue='gender', palette='magma', legend= False)
plt.title('Waiting Days by Gender')
plt.xlabel('Gender (0=Female, 1=Male)')
plt.ylabel('Waiting Days')
plt.show()

  # Age vs Waiting Days 
plt.figure(figsize=(8, 5))
plt.scatter(df['age_yrs'], df['waiting_days'], alpha=0.3, color='lightpink')
plt.title('Age vs Waiting Days')
plt.xlabel('Age (years)')
plt.ylabel('Waiting Days')
plt.show()

print(f"Correlation between age and waiting days: {df['age_yrs'].corr(df['waiting_days']):.2f}")

  # By ASA Score 
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='asa_score', y='waiting_days',  hue='asa_score', palette='magma', legend= False)
plt.title('Waiting Days by ASA Score')
plt.xlabel('ASA Score')
plt.ylabel('Waiting Days')
plt.show()

print("\nMedian waiting days per ASA score:")
print(df.groupby('asa_score')['waiting_days'].median())

INSIGHT: None of these features independently predict wait time. Procedure type differences are confounded by cat_referral distribution. Gender shows negligibledifference. Age is essentially uncorrelated (r=0.02) - scheduling is age-blind. ASA score shows a mild inverse trend (higher ASA -> slightly shorter wait) but this is likely mediated through cat_referral assignment, not an independent effect.

In [ ]:
# Which features correlate most strongly with waiting_days?
numeric_cols = df.select_dtypes(include=np.number)
target_corr = numeric_cols.corr()['waiting_days'].sort_values(ascending=False)
strong_corr = target_corr[abs(target_corr) > 0.2]

plt.figure(figsize=(6,6))
sns.heatmap(strong_corr.to_frame(), annot=True, cmap='RdPu')

INSIGHT: cat_referral (+0.77) and recommended_max_wait_days (+0.75) are near-collinear. Use one or the other. risk_score (-0.71) is next strongest. ind_fobt (-0.40) and nbcsp (-0.43) reflect urgency-driven fast-tracking.

In [ ]:
# Does a rolling average reveal a cleaner trend? Does log-transform help?

df['referral_date'] = pd.to_datetime(df['referral_date'], errors='coerce')
df = df.dropna(subset=['referral_date'])

df = df.sort_values('referral_date')

monthly_avg = df.set_index('referral_date')['waiting_days'].resample('ME').mean()

monthly_avg_rolling = monthly_avg.rolling(window=3).mean()

plt.figure(figsize=(10,5))
monthly_avg_rolling.plot(color='lightpink')
plt.title("3-Month Rolling Average Waiting Time")
plt.ylabel("Days")

df['log_waiting_days'] = np.log1p(df['waiting_days'])

plt.figure(figsize=(6,4))
sns.histplot(df['log_waiting_days'], kde=True, color ="#E0218A" )
plt.title("Log-Transformed Waiting Time Distribution")
plt.xlabel("Log Waiting Days")

INSIGHT: Rolling average confirms long-run stability after the 2010 spike. Log-transform gives near-symmetric distribution - good for MLP/linear models, not needed for trees.

In [ ]:
# What fraction of each category experiences a "long wait"?
df['long_wait'] = df['waiting_days'] > df['recommended_max_wait_days']

sns.barplot(data=df, x='cat_referral_label', y='long_wait', color = "lightpink")
plt.xticks(rotation=45)
plt.title("Proportion of Long Waits by Category")

INSIGHT: Cat 3 dominates, Cat 2 represents the highest operational leverage point, as it combines high volume with the highest breach rate.

### Conclusion : Waiting Time 

Final Insights: Waiting Time Analysis

**Distributional Properties**

`waiting_days` is positively skewed (skewness = 1.84), with mean = 52.2, median = 46.0, and std = 44.2 days (range: 1-296). The IQR spans 18-71 days. Median-based statistics and non-parametric methods are more appropriate than mean-based comparisons for this feature.

**Stratification by Referral Category**

Referral category acts as a near-deterministic partitioner of waiting time:

| Category | n | Median (days) | Mean (days) | Std |
|---|---|---|---|---|
| Urgent (Cat 1) | 550 | 15 | 15.2 | 7.0 |
| Semi-Urgent (Cat 2) | 961 | 59 | 58.2 | 22.7 |
| Routine (Cat 3) | 171 | 137 | 137.8 | 64.9 |

Cat 1's low std (7.0) reflects tightly controlled urgent scheduling. Cat 3's high variance (64.9) reflects genuinely flexible windows, not poor performance.

**Threshold Breach Analysis**

81 patients (4.8%) exceeded their `recommended_max_wait_days`. Semi-Urgent (Cat 2) has the highest breach rate at **7.6%** despite being the largest group (n=961), pointing to systemic capacity strain in this tier. Cat 1 sits at 1.5%, near-compliant. Cat 3's 0% is an artefact of its generous threshold, not operational efficiency.

**Correlation with Other Features**

Age (r≈0), gender, and ASA score show no meaningful relationship with waiting days. Scheduling appears to be driven almost entirely by referral urgency. Procedure type differences are likely confounded by cat_referral distribution rather than representing an independent effect.

**Modelling Implications**

- `cat_referral` will likely dominate feature importance. Encoding choice (ordinal vs. one-hot) matters.
- `recommended_max_wait_days` is near-collinear with `cat_referral` (r=0.75). Use one or the other in linear models.
- The skewed distribution favours tree-based models. If using MLP or linear approaches, log-transform `waiting_days` first.
- The Semi-Urgent cohort warrants the most attention. Highest breach rate combined with the largest patient volume means the greatest clinical impact.

### Procedure Type

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='procedure_type', data=df)
plt.title("Procedure Type Distribution")
plt.show()

print("Count of each procedure type:")
print(df['procedure_type'].value_counts())
print("\nProportion of each procedure type:")
print(df['procedure_type'].value_counts(normalize=True))

The dataset is largely composed of colonoscopy cases (81%), whereas pan-endoscopy and flexible sigmoidoscopy are comparatively rare.

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x='procedure_type', y='risk_score', data=df)
plt.title("Risk Score by Procedure Type")
plt.show()

print("Average Risk Score by Procedure Type:")
df.groupby('procedure_type')['risk_score'].mean().sort_values(ascending=False)

Risk scores are similarly distributed across all procedure types, with comparable medians and interquartile ranges. This suggests that procedure type alone is not a strong predictor of patient risk. While slightly higher maximum risk scores are observed in colonoscopy and flexible sigmoidoscopy, the overall differences remain minimal.

In [ ]:
pd.crosstab(df['procedure_type'], df['cat_referral'])
pd.crosstab(df['procedure_type'], df['cat_referral']).plot(kind='bar', stacked=True)
plt.xticks(rotation= 5)
plt.title("Procedure Type vs Referral Category")
plt.show()

ct = pd.crosstab(df['procedure_type'], df['cat_referral'])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
print("Proportion of cat_referral within each procedure_type:\n")
print(ct_pct)

The proportions of referral categories within each procedure type show a similar pattern.

In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(data=df, x='procedure_type', y='waiting_days')
plt.title('Waiting Days by Procedure Type')
plt.xlabel('Procedure Type')
plt.ylabel('Waiting Days')
plt.xticks(rotation=20)
plt.show()

Waiting times are broadly similar across all procedure types, with comparable medians and interquartile ranges. However, colonoscopy exhibits a significantly higher number of extreme outliers, indicating that a subset of patients experience substantial delays. Flexible sigmoidoscopy also shows occasional extreme delays, while pan-endoscopy appears relatively more stable.

### Distribution
The dataset is heavily dominated by colonoscopy procedures, with flexible sigmoidoscopy and pan-endoscopy making up a much smaller proportion. 

This imbalance is important to keep in mind, as any patterns observed especially in waiting times may be influenced by the higher volume of colonoscopy cases rather than the procedure itself.

### Procedure Type and Referral Category
Across all procedure types, most cases fall under Category 2. This is particularly noticeable for colonoscopy, which has the largest number of semi-urgent referrals.

This suggests that demand is concentrated in mid-priority cases, which may contribute to increased pressure on the system.

### Risk Score by Procedure Type
The distribution of risk scores is very similar across all procedure types. The medians are almost identical (around 0.27 to 0.28), and the spread of the data is also consistent.

This indicates that procedure type does not have a strong relationship with patient risk. In other words, patients undergoing different procedures tend to have similar risk levels.

### Waiting Time by Procedure Type
Median waiting times are also quite similar across all procedure types, generally falling between 40 and 50 days. Most patients are seen within a similar range (around 20 to 70 days).

However, there is a noticeable difference in variability:
- Colonoscopy has a large number of extreme waiting time outliers (over 150 days)
- Flexible sigmoidoscopy shows a few very high delays (up to around 300 days)
- Pan-endoscopy appears more stable with fewer extreme cases

This suggests that while typical waiting times are similar, some procedures (particularly colonoscopy) are more affected by long delays.

### Conclusion
Overall, procedure type does not appear to strongly influence either risk score or typical waiting time. The main differences are seen in the number of extreme delays, especially for colonoscopy.

This implies that delays are more likely caused by system-level factors, such as demand and capacity, rather than the type of procedure itself. As a result, procedure type alone may not be a very useful feature for predicting risk or waiting time, and other variables should be considered.

These findings directly informed model selection, favouring ensemble methods capable of capturing non-linear interactions and prioritising recall-based evaluation metrics for clinically critical cases.

## Binary Variable Definitions

All binary (0/1) indicator columns used as features in this model are defined below.

---

### Clinical Indication Flags (`ind_*`)

Each `ind_*` column is 1 if the patient was referred for that clinical indication, 0 otherwise. A patient may have multiple indications simultaneously.

| Column | Clinical Meaning |
|---|---|
| `ind_therapeutic` | Procedure is therapeutic (e.g., polypectomy, haemostasis) rather than purely diagnostic |
| `ind_pr_bleed` | Per-rectal (PR) bleeding — fresh blood passed rectally |
| `ind_melaena` | Melaena — dark, tarry stools indicating upper gastrointestinal bleeding |
| `ind_fobt` | Positive faecal occult blood test (FOBT / NBCSP bowel cancer screening result) |
| `ind_anaemia` | Unexplained iron-deficiency anaemia |
| `ind_polyp_surveillance` | Surveillance colonoscopy following prior polyp removal |
| `ind_ibd` | Inflammatory bowel disease monitoring (Crohn's disease or ulcerative colitis) |
| `ind_ca_surveillance` | Cancer surveillance — follow-up after treated colorectal cancer |
| `ind_family_hx` | Family history of colorectal cancer (elevated hereditary risk) |
| `ind_other` | Any other clinical indication not captured above |

**Clinical urgency note:** `ind_melaena`, `ind_pr_bleed`, and `ind_fobt` are the strongest signals for Category 1 (urgent) referrals. `ind_polyp_surveillance`, `ind_ibd`, and `ind_ca_surveillance` are more commonly associated with Category 2 or 3 (semi-urgent / routine).

---

### Referral Source Flags (`ref_*`)

Each `ref_*` column is 1 if the patient was referred from that source, 0 otherwise. Exactly one `ref_*` flag is 1 per patient.

| Column | Referral Source |
|---|---|
| `ref_gp` | General Practitioner (community GP) |
| `ref_priv_med` | Private medical specialist |
| `ref_priv_surg` | Private surgical specialist |
| `ref_op_med` | Outpatient medical department |
| `ref_op_surg` | Outpatient surgical department |
| `ref_inpt_med` | Inpatient medical ward |
| `ref_inpt_surg` | Inpatient surgical ward |
| `ref_other` | Other referral source |

**Clinical note:** Inpatient referrals (`ref_inpt_med`, `ref_inpt_surg`) tend to be higher urgency as the patient is already hospitalised. GP referrals span all urgency categories and represent the largest volume.

---

### Binary Target Variable (`exceeded_wait`)

| Value | Meaning |
|---|---|
| 0 | Patient waited ≤ `recommended_max_wait_days` (within target) |
| 1 | Patient waited > `recommended_max_wait_days` (breached recommended wait time) |

Derived as: `exceeded_wait = (waiting_days > recommended_max_wait_days)`.

**Class imbalance:** Only ~4–6% of patients breach their recommended wait time, making this a heavily imbalanced binary classification task.

---

### Other Binary Columns Used

| Column | Meaning |
|---|---|
| `index_scope` | 1 if this is the patient's first (index) endoscopy; 0 if repeat |
| `surveillance_scope` | 1 if this is a surveillance endoscopy; 0 otherwise |
| `nbcsp` | 1 if the referral originates from the National Bowel Cancer Screening Program |

In [ ]:
# Baseline classification target: cat_referral (Cat 1 / Cat 2 / Cat 3)
# Keep the feature set leakage-safe by using only pre-procedure demographics,
# referral flags, and indication flags.

target_col = "cat_referral"

allowed_direct_features = [
    "age_yrs",
    "gender",
    "bmi",
    "asa_score",
    "procedure_type",
    "index_scope",
    "surveillance_scope",
    "nbcsp",
  
]
allowed_prefixes = ("ref_", "ind_")

feature_cols = [
    column
    for column in df.columns
    if column in allowed_direct_features or column.startswith(allowed_prefixes)
]

excluded_columns = [
    column
    for column in [
        "patient_id",
        "referral_date",
        "procedure_date",
        "waiting_days",
        "cat_referral",
        "recommended_max_wait_days",
        "successful_procedure",
        "status",
        "endoscopist",
        "procedure_room",
        "complication",
        "complication_type",
        "ti_intubation",
        "finding_normal",
        "finding_no_polyps",
        "finding_polyps",
        "finding_diverticular",
        "finding_inflam_ulcer",
        "finding_mass",
        "finding_angiodysplasia",
        "finding_stricture",
        "finding_haemorrhoids",
        "finding_other",
        "follow_up_months",
        "risk_score",
    ]
    if column in df.columns
]

model_df = df[feature_cols + [target_col]].copy()
model_df = model_df.dropna(subset=[target_col]).reset_index(drop=True)

X = model_df.drop(columns=[target_col]).copy()
y = model_df[target_col].astype(int).copy()

categorical_cols = [col for col in ["procedure_type", "index_scope", "surveillance_scope"] if col in X.columns]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

for column in categorical_cols:
    X[column] = X[column].astype("string")

print("Selected feature columns:\n", feature_cols)
print("\nExcluded columns:\n", excluded_columns)
print("\nTarget distribution:")
print(y.value_counts().sort_index())
print(y.value_counts(normalize=True).sort_index().round(3))

### Baseline Models

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
LABELS = [1, 2, 3]
TARGET_NAMES = ["Cat 1", "Cat 2", "Cat 3"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            categorical_cols,
        ),
    ],
    remainder="drop",
)


def cat1_recall_score(y_true, y_pred):
    return recall_score((np.asarray(y_true) == 1).astype(int), (np.asarray(y_pred) == 1).astype(int), zero_division=0)


scoring = {
    "cat1_recall": make_scorer(cat1_recall_score),
    "macro_f1": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
}


def make_pipeline(model):
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", model),
        ]
    )


models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=RANDOM_STATE,
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "SVM": SVC(
        kernel="rbf",
        C=2.0,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE,
    ),
    "KNN": KNeighborsClassifier(n_neighbors=25),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []
fitted_models = {}

for model_name, estimator in models.items():
    pipeline = make_pipeline(estimator)
    cv_scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    fitted_models[model_name] = pipeline
    results.append(
        {
            "model": model_name,
            "cv_cat1_recall": cv_scores["test_cat1_recall"].mean(),
            "cv_macro_f1": cv_scores["test_macro_f1"].mean(),
            "cv_balanced_accuracy": cv_scores["test_balanced_accuracy"].mean(),
            "test_cat1_recall": recall_score((y_test == 1).astype(int), (y_pred == 1).astype(int), zero_division=0),
            "test_macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
            "test_balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_precision_cat1": precision_score((y_test == 1).astype(int), (y_pred == 1).astype(int), zero_division=0),
        }
    )

results_df = pd.DataFrame(results).sort_values(
    by=["test_cat1_recall", "test_macro_f1", "test_balanced_accuracy"],
    ascending=False,
).reset_index(drop=True)

display(results_df)

best_model_name = results_df.loc[0, "model"]
best_model = fitted_models[best_model_name]
best_predictions = best_model.predict(X_test)

print(f"Best baseline model: {best_model_name}")
print("\nClassification report:")
print(
    classification_report(
        y_test,
        best_predictions,
        labels=LABELS,
        target_names=TARGET_NAMES,
        zero_division=0,
    )
)

cm = confusion_matrix(y_test, best_predictions, labels=LABELS)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=TARGET_NAMES,
    yticklabels=TARGET_NAMES,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix: {best_model_name}")
plt.tight_layout()
plt.show()

feature_names = best_model.named_steps["preprocess"].get_feature_names_out()
final_estimator = best_model.named_steps["model"]

if hasattr(final_estimator, "feature_importances_"):
    importance_df = (
        pd.DataFrame(
            {
                "feature": feature_names,
                "importance": final_estimator.feature_importances_,
            }
        )
        .sort_values("importance", ascending=False)
        .head(15)
    )
    display(importance_df)
elif hasattr(final_estimator, "coef_"):
    coef_df = pd.DataFrame(final_estimator.coef_, columns=feature_names, index=TARGET_NAMES)
    display(coef_df.T.sort_values(by="Cat 1", ascending=False).head(15))

In [ ]:
# Confusion matrices for every baseline model
n_models = len(fitted_models)
ncols = 4
nrows = -(-n_models // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4))
axes = axes.flatten()

for i, (model_name, pipeline) in enumerate(fitted_models.items()):
    y_pred_i = pipeline.predict(X_test)
    cm_i = confusion_matrix(y_test, y_pred_i, labels=LABELS)
    sns.heatmap(
        cm_i,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=TARGET_NAMES,
        yticklabels=TARGET_NAMES,
        ax=axes[i],
    )
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")
    cat1_rec = recall_score((y_test == 1).astype(int), (y_pred_i == 1).astype(int), zero_division=0)
    axes[i].set_title(f"{model_name} Cat 1 Recall: {cat1_rec:.1%}")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Confusion Matrices — All Baseline Models", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Enhanced Models

## Final Summary: Category Prediction

This notebook predicts `cat_referral` (Cat 1, Cat 2, Cat 3) using pre-procedure information, with a strong emphasis on protecting urgent patients by maximizing Cat 1 recall.

### Ranking Approach

Models are ranked by:
1. **Macro F2** (emphasizes recall over precision, crucial for not missing urgent cases)
2. **Cat 1 Recall** (ensures urgent cases are identified)
3. **Macro F1** (overall balance)

This ranking criteria aligns with the clinical goal: minimizing missed urgent referrals while maintaining strong multiclass performance.

### Key Finding: Ensemble Improvement

When all candidate models are compared, the **soft-voting ensemble** ranks first under the unified ranking criteria above.

This is a notable finding because:
- SVM remains the best single model and was not grid-searched in the original baseline comparison
- The ensemble combines the strongest fitted models and improves the recall-weighted score slightly
- Cat 1 recall is preserved while macro F2 and macro F1 improve
- A symptom-augmented ensemble was also tested, which explicitly adds symptom burden features, but it did not beat the equal-weight ensemble

### Model Performance Summary

- **Best baseline model:** SVM (94.62% accuracy, 97.27% Cat 1 recall)
- **Best among tuned models:** LightGBM (94.07% accuracy, 97.27% Cat 1 recall, 87.27% balanced accuracy)
- **Overall winner (unified ranking):** Soft-voting ensemble (macro F2 + Cat 1 recall + macro F1)

### Takeaway

The soft-voting ensemble is the final selected model for clinical deployment because it provides the best recall-weighted balance under the ranking criteria. SVM remains the strongest single model, and the symptom-augmented ensemble confirms that symptom information can be incorporated explicitly, even though it does not outperform the simpler equal-weight ensemble.

In [ ]:
def _compute_all_metrics(model_name, y_true, y_pred, group_name):
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(y_pred)

    cat1_true = (y_true_arr == 1).astype(int)
    cat1_pred = (y_pred_arr == 1).astype(int)

    return {
        "group": group_name,
        "model": model_name,
        "accuracy": accuracy_score(y_true_arr, y_pred_arr),
        "balanced_accuracy": balanced_accuracy_score(y_true_arr, y_pred_arr),
        "macro_precision": precision_score(y_true_arr, y_pred_arr, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true_arr, y_pred_arr, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true_arr, y_pred_arr, average="macro", zero_division=0),
        "macro_f2": fbeta_score(y_true_arr, y_pred_arr, beta=2, average="macro", zero_division=0),
        "cat1_precision": precision_score(cat1_true, cat1_pred, zero_division=0),
        "cat1_recall": recall_score(cat1_true, cat1_pred, zero_division=0),
        "cat1_f2": fbeta_score(cat1_true, cat1_pred, beta=2, zero_division=0),
        "mse": mean_squared_error(y_true_arr, y_pred_arr),
        "mae": mean_absolute_error(y_true_arr, y_pred_arr),
    }


comparison_rows = []

# Baseline models
for model_name, model_pipeline in fitted_models.items():
    baseline_pred = model_pipeline.predict(X_test)
    comparison_rows.append(_compute_all_metrics(model_name, y_test, baseline_pred, "Baseline"))

# Threshold-tuned model
threshold_model_name = globals().get("threshold_model_name", "Threshold-Tuned Model")
best_threshold = globals().get("best_threshold", None)
tuned_test_predictions = globals().get("tuned_test_predictions", None)

# Fallback: use an existing tuned prediction if threshold outputs were not created
if tuned_test_predictions is None and "model_predictions" in globals():
    model_predictions_dict = globals().get("model_predictions", {})
    if isinstance(model_predictions_dict, dict):
        tuned_test_predictions = model_predictions_dict.get("Tuned Logistic Regression")
    if tuned_test_predictions is not None and best_threshold is None:
        best_threshold = 0.50
        threshold_model_name = "Tuned Logistic Regression"

if tuned_test_predictions is not None:
    threshold_text = (
        f"threshold={best_threshold:.2f}"
        if isinstance(best_threshold, (int, float, np.floating))
        else "threshold=NA"
    )
    comparison_rows.append(
        _compute_all_metrics(
            f"{threshold_model_name} ({threshold_text})",
            y_test,
            tuned_test_predictions,
            "Threshold Tuned",
        )
    )
else:
    print("Threshold-tuned predictions not found; skipped Threshold Tuned row.")

# Enhanced tree models
enhanced_added = False

# Prefer fitted enhanced pipelines if that cell has been run
enhanced_models_dict = globals().get("enhanced_fitted_models", {})
if isinstance(enhanced_models_dict, dict) and len(enhanced_models_dict) > 0:
    for model_name, model_pipeline in enhanced_models_dict.items():
        enhanced_pred = model_pipeline.predict(X_test)
        comparison_rows.append(_compute_all_metrics(model_name, y_test, enhanced_pred, "Enhanced"))
        enhanced_added = True
else:
    # Fallback: use stored predictions if enhanced_fitted_models is unavailable
    pred_store = globals().get("model_predictions", {})
    if not isinstance(pred_store, dict) or len(pred_store) == 0:
        pred_store = globals().get("model_predictions_dict", {})

    for enhanced_name in ["Extra Trees", "Tuned Random Forest", "Tuned XGBoost", "Tuned LightGBM"]:
        preds = pred_store.get(enhanced_name) if isinstance(pred_store, dict) else None
        if preds is not None and len(preds) == len(y_test):
            comparison_rows.append(_compute_all_metrics(enhanced_name, y_test, preds, "Enhanced"))
            enhanced_added = True

if not enhanced_added:
    print("Enhanced models not found; skipped Enhanced rows.")

all_results_df = pd.DataFrame(comparison_rows).sort_values(
    by=["macro_f2", "cat1_recall", "macro_f1"],
    ascending=False,
).reset_index(drop=True)

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "macro_f2",
    "cat1_precision",
    "cat1_recall",
    "cat1_f2",
    "mse",
    "mae",
]

all_results_df[metric_columns] = all_results_df[metric_columns].round(4)
display(all_results_df)

print("Best model by macro F2:", all_results_df.loc[0, "model"])

In [ ]:
# Model Exploration and Optimisation

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, precision_score, classification_report
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scoring dict
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": "f1_macro",
    "macro_recall": "recall_macro"
}

# Accumulates best-params rows; each GridSearch cell appends to this
hyperparameter_rows = []

In [ ]:
# Tuned Logistic Regression

log_reg_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

log_reg_params = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["lbfgs", "liblinear"]
}

log_reg_grid = GridSearchCV(
    log_reg_pipeline,
    log_reg_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)

log_reg_grid.fit(X_train, y_train)

best_log_reg = log_reg_grid.best_estimator_
y_pred_log = best_log_reg.predict(X_test)

print("Best Logistic Regression Params:")
print(log_reg_grid.best_params_)

print("\nTuned Logistic Regression Results:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_log), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_log), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_log, average="macro"), 4))
print("Macro Recall:", round(recall_score(y_test, y_pred_log, average="macro"), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))

hyperparameter_rows.append({
    "Model": "Logistic Regression",
    "Type": "grid-search best",
    "Best CV Score (macro F1)": round(log_reg_grid.best_score_, 4),
    "Optimal Parameters": "; ".join(f"{k}={v}" for k, v in sorted(log_reg_grid.best_params_.items())),
})

In [ ]:
# Tuned Random Forest
rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

rf_params = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print("Best Random Forest Params:")
print(rf_grid.best_params_)

print("\nTuned Random Forest Results:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_rf), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_rf), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_rf, average="macro"), 4))
print("Macro Recall:", round(recall_score(y_test, y_pred_rf, average="macro"), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

hyperparameter_rows.append({
    "Model": "Random Forest",
    "Type": "grid-search best",
    "Best CV Score (macro F1)": round(rf_grid.best_score_, 4),
    "Optimal Parameters": "; ".join(f"{k}={v}" for k, v in sorted(rf_grid.best_params_.items())),
})

In [ ]:
# Extra Trees Model
extra_trees_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", ExtraTreesClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight="balanced",
        random_state=42
    ))
])

extra_trees_pipeline.fit(X_train, y_train)
y_pred_extra = extra_trees_pipeline.predict(X_test)

print("Extra Trees Results:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_extra), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_extra), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_extra, average="macro"), 4))
print("Macro Recall:", round(recall_score(y_test, y_pred_extra, average="macro"), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_extra))

In [ ]:
# Tuned Extra Trees — GridSearchCV

et_gs_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
])

et_params = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
}

et_grid = GridSearchCV(
    et_gs_pipeline,
    et_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)
et_grid.fit(X_train, y_train)

best_et = et_grid.best_estimator_
y_pred_et_tuned = best_et.predict(X_test)

print("Best Extra Trees Params:")
print(et_grid.best_params_)

print("\nTuned Extra Trees Results:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_et_tuned), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_et_tuned), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_et_tuned, average='macro'), 4))
print("Cat 1 Recall:", round(recall_score(y_test, y_pred_et_tuned, labels=[1], average='macro'), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_et_tuned, labels=LABELS, target_names=TARGET_NAMES, zero_division=0))

hyperparameter_rows.append({
    "Model": "Extra Trees",
    "Type": "grid-search best",
    "Best CV Score (macro F1)": round(et_grid.best_score_, 4),
    "Optimal Parameters": "; ".join(f"{k}={v}" for k, v in sorted(et_grid.best_params_.items())),
})

In [ ]:
# Tuned Decision Tree — GridSearchCV

dt_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
])

dt_params = {
    "model__max_depth": [None, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__criterion": ["gini", "entropy"],
}

dt_grid = GridSearchCV(
    dt_pipeline,
    dt_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)
dt_grid.fit(X_train, y_train)

best_dt = dt_grid.best_estimator_
y_pred_dt_tuned = best_dt.predict(X_test)

print("Best Decision Tree Params:")
print(dt_grid.best_params_)

print("\nTuned Decision Tree Results:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_dt_tuned), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_dt_tuned), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_dt_tuned, average='macro'), 4))
print("Cat 1 Recall:", round(recall_score(y_test, y_pred_dt_tuned, labels=[1], average='macro'), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt_tuned, labels=LABELS, target_names=TARGET_NAMES, zero_division=0))

hyperparameter_rows.append({
    "Model": "Decision Tree",
    "Type": "grid-search best",
    "Best CV Score (macro F1)": round(dt_grid.best_score_, 4),
    "Optimal Parameters": "; ".join(f"{k}={v}" for k, v in sorted(dt_grid.best_params_.items())),
})

In [ ]:
# Tuned KNN — GridSearchCV

knn_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", KNeighborsClassifier()),
])

knn_params = {
    "model__n_neighbors": [5, 11, 15, 21, 25, 31],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
}

knn_grid = GridSearchCV(
    knn_pipeline,
    knn_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)
knn_grid.fit(X_train, y_train)

best_knn = knn_grid.best_estimator_
y_pred_knn_tuned = best_knn.predict(X_test)

print("Best KNN Params:")
print(knn_grid.best_params_)

print("\nTuned KNN Results:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_knn_tuned), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_knn_tuned), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_knn_tuned, average='macro'), 4))
print("Cat 1 Recall:", round(recall_score(y_test, y_pred_knn_tuned, labels=[1], average='macro'), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn_tuned, labels=LABELS, target_names=TARGET_NAMES, zero_division=0))

hyperparameter_rows.append({
    "Model": "KNN",
    "Type": "grid-search best",
    "Best CV Score (macro F1)": round(knn_grid.best_score_, 4),
    "Optimal Parameters": "; ".join(f"{k}={v}" for k, v in sorted(knn_grid.best_params_.items())),
})

In [ ]:
# Tuned Gradient Boosting — GridSearchCV

gb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
])

gb_params = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.05, 0.1, 0.2],
    "model__min_samples_split": [2, 5],
}

gb_grid = GridSearchCV(
    gb_pipeline,
    gb_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)
gb_grid.fit(X_train, y_train)

best_gb = gb_grid.best_estimator_
y_pred_gb_tuned = best_gb.predict(X_test)

print("Best Gradient Boosting Params:")
print(gb_grid.best_params_)

print("\nTuned Gradient Boosting Results:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_gb_tuned), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_gb_tuned), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_gb_tuned, average='macro'), 4))
print("Cat 1 Recall:", round(recall_score(y_test, y_pred_gb_tuned, labels=[1], average='macro'), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb_tuned, labels=LABELS, target_names=TARGET_NAMES, zero_division=0))

hyperparameter_rows.append({
    "Model": "Gradient Boosting",
    "Type": "grid-search best",
    "Best CV Score (macro F1)": round(gb_grid.best_score_, 4),
    "Optimal Parameters": "; ".join(f"{k}={v}" for k, v in sorted(gb_grid.best_params_.items())),
})

In [ ]:
# Tuned SVM — GridSearchCV

svm_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", SVC(class_weight="balanced", random_state=RANDOM_STATE)),
])

svm_params = {
    "model__C": [0.1, 1, 2, 5, 10],
    "model__kernel": ["rbf", "poly"],
    "model__gamma": ["scale", "auto"],
}

svm_grid = GridSearchCV(
    svm_pipeline,
    svm_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)
svm_grid.fit(X_train, y_train)

best_svm = svm_grid.best_estimator_
y_pred_svm_tuned = best_svm.predict(X_test)

print("Best SVM Params:")
print(svm_grid.best_params_)

print("\nTuned SVM Results:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_svm_tuned), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_svm_tuned), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_svm_tuned, average='macro'), 4))
print("Cat 1 Recall:", round(recall_score(y_test, y_pred_svm_tuned, labels=[1], average='macro'), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm_tuned, labels=LABELS, target_names=TARGET_NAMES, zero_division=0))

hyperparameter_rows.append({
    "Model": "SVM",
    "Type": "grid-search best",
    "Best CV Score (macro F1)": round(svm_grid.best_score_, 4),
    "Optimal Parameters": "; ".join(f"{k}={v}" for k, v in sorted(svm_grid.best_params_.items())),
})

In [ ]:
# XGBoost tuning
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

xgb_label_encoder = LabelEncoder()
y_train_xgb = xgb_label_encoder.fit_transform(y_train)
y_test_xgb = xgb_label_encoder.transform(y_test)

xgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
    )),
])

xgb_params = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [3, 4, 5],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
}

xgb_grid = GridSearchCV(
    xgb_pipeline,
    xgb_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)

xgb_grid.fit(X_train, y_train_xgb)

best_xgb = xgb_grid.best_estimator_
y_pred_xgb = xgb_label_encoder.inverse_transform(best_xgb.predict(X_test))

print("Best XGBoost Params:")
print(xgb_grid.best_params_)

print("\nTuned XGBoost Results:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_xgb), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_xgb), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_xgb, average="macro"), 4))
print("Macro Recall:", round(recall_score(y_test, y_pred_xgb, average="macro"), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, labels=LABELS, target_names=TARGET_NAMES, zero_division=0))

hyperparameter_rows.append({
    "Model": "XGBoost",
    "Type": "grid-search best",
    "Best CV Score (macro F1)": round(xgb_grid.best_score_, 4),
    "Optimal Parameters": "; ".join(f"{k}={v}" for k, v in sorted(xgb_grid.best_params_.items())),
})

In [ ]:
# LightGBM tuning
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV

lgbm_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LGBMClassifier(
        objective="multiclass",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

lgbm_params = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [-1, 8, 12],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__num_leaves": [31, 63],
    "model__min_child_samples": [10, 20],
}

lgbm_grid = GridSearchCV(
    lgbm_pipeline,
    lgbm_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
)

lgbm_grid.fit(X_train, y_train)

best_lgbm = lgbm_grid.best_estimator_
y_pred_lgbm = best_lgbm.predict(X_test)

print("Best LightGBM Params:")
print(lgbm_grid.best_params_)

print("\nTuned LightGBM Results:")
print("Accuracy:", round(accuracy_score(y_test, y_pred_lgbm), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, y_pred_lgbm), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_lgbm, average="macro"), 4))
print("Macro Recall:", round(recall_score(y_test, y_pred_lgbm, average="macro"), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lgbm, labels=LABELS, target_names=TARGET_NAMES, zero_division=0))

hyperparameter_rows.append({
    "Model": "LightGBM",
    "Type": "grid-search best",
    "Best CV Score (macro F1)": round(lgbm_grid.best_score_, 4),
    "Optimal Parameters": "; ".join(f"{k}={v}" for k, v in sorted(lgbm_grid.best_params_.items())),
})

### Unified Ranking and Final Model Selection


This table ranks every tuned model on the same held-out test set so the best performer can be selected objectively.

**What is in the table**
All models that went through hyperparameter tuning (GridSearchCV with StratifiedKFold) are included: Logistic Regression, Random Forest, Extra Trees, Decision Tree, KNN, Gradient Boosting, SVM, XGBoost, and LightGBM. The Extra Trees entry uses fixed high-performing parameters found during exploration rather than a full grid search.

**Ranking criteria**
Models are sorted by three metrics in priority order:
1. **Cat 1 Recall** (primary) — the proportion of urgent referrals correctly identified. Missing a Cat 1 patient is the most dangerous error in this setting, so this drives the ranking.
2. **Macro F1** (secondary) — balances precision and recall across all three categories. A high Cat 1 recall at the expense of Cat 2/3 accuracy is not acceptable, so Macro F1 breaks ties.
3. **Balanced Accuracy** (tertiary) — accounts for class imbalance; used as a final tie-breaker.

**Final selection**
The model ranked first in this table is selected as the primary recommendation. Cat 2 and Cat 3 recall columns are shown alongside Cat 1 so any model that gains Cat 1 recall by sacrificing the other categories is visible and can be penalised accordingly.


In [ ]:
# Final Optimized Model Comparison (Tuned/Enhanced Models Only - baseline SVM excluded)

model_predictions = {
    "Tuned Logistic Regression": y_pred_log,
    "Tuned Random Forest": y_pred_rf,
    "Extra Trees (fixed params)": y_pred_extra,
}

for var_name, label in [
    ("y_pred_et_tuned",  "Tuned Extra Trees"),
    ("y_pred_dt_tuned",  "Tuned Decision Tree"),
    ("y_pred_knn_tuned", "Tuned KNN"),
    ("y_pred_gb_tuned",  "Tuned Gradient Boosting"),
    ("y_pred_svm_tuned", "Tuned SVM"),
    ("y_pred_xgb",       "Tuned XGBoost"),
    ("y_pred_lgbm",      "Tuned LightGBM"),
]:
    if var_name in globals():
        model_predictions[label] = globals()[var_name]

comparison_rows = []

for model_name, preds in model_predictions.items():
    comparison_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, preds),
        "Balanced Accuracy": balanced_accuracy_score(y_test, preds),
        "Macro F1": f1_score(y_test, preds, average="macro"),
        "Macro Recall": recall_score(y_test, preds, average="macro"),
        "Cat 1 Recall": recall_score((y_test == 1).astype(int), (preds == 1).astype(int), zero_division=0),
        "Cat 1 Precision": precision_score((y_test == 1).astype(int), (preds == 1).astype(int), zero_division=0),
        "Cat 2 Recall": recall_score((y_test == 2).astype(int), (preds == 2).astype(int), zero_division=0),
        "Cat 3 Recall": recall_score((y_test == 3).astype(int), (preds == 3).astype(int), zero_division=0),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    by=["Cat 1 Recall", "Macro F1", "Balanced Accuracy"],
    ascending=False
)

display(comparison_df.round(4))

print("\nNote: This table shows tuned/enhanced models only. For comparison with baseline SVM, see the Unified Comparison table above.")

In [ ]:
print("=" * 80)
print("HYPERPARAMETER SUMMARY WITH ACCURACY IMPROVEMENT")
print("=" * 80)

from sklearn.metrics import accuracy_score

# Map tuned models to their GridSearch objects, tuned predictions, and baseline model names
# used for before-vs-after accuracy comparison.
tuned_model_map = [
    {
        "Model": "Logistic Regression",
        "grid_obj": globals().get("log_reg_grid"),
        "pred": globals().get("y_pred_log"),
        "baseline_name": "Logistic Regression",
    },
    {
        "Model": "Random Forest",
        "grid_obj": globals().get("rf_grid"),
        "pred": globals().get("y_pred_rf"),
        "baseline_name": "Random Forest",
    },
    {
        "Model": "Extra Trees",
        "grid_obj": globals().get("et_grid"),
        "pred": globals().get("y_pred_et_tuned"),
        "baseline_name": None,
    },
    {
        "Model": "Decision Tree",
        "grid_obj": globals().get("dt_grid"),
        "pred": globals().get("y_pred_dt_tuned"),
        "baseline_name": "Decision Tree",
    },
    {
        "Model": "KNN",
        "grid_obj": globals().get("knn_grid"),
        "pred": globals().get("y_pred_knn_tuned"),
        "baseline_name": "KNN",
    },
    {
        "Model": "Gradient Boosting",
        "grid_obj": globals().get("gb_grid"),
        "pred": globals().get("y_pred_gb_tuned"),
        "baseline_name": "Gradient Boosting",
    },
    {
        "Model": "SVM",
        "grid_obj": globals().get("svm_grid"),
        "pred": globals().get("y_pred_svm_tuned"),
        "baseline_name": "SVM",
    },
    {
        "Model": "XGBoost",
        "grid_obj": globals().get("xgb_grid"),
        "pred": globals().get("y_pred_xgb"),
        "baseline_name": None,
    },
    {
        "Model": "LightGBM",
        "grid_obj": globals().get("lgbm_grid"),
        "pred": globals().get("y_pred_lgbm"),
        "baseline_name": None,
    },
]

baseline_pool = globals().get("fitted_models", {})
hyperparam_summary_rows = []

for item in tuned_model_map:
    grid_obj = item["grid_obj"]
    tuned_pred = item["pred"]

    if grid_obj is None or tuned_pred is None:
        continue

    best_params = grid_obj.best_params_ if hasattr(grid_obj, "best_params_") else {}
    tuned_acc = accuracy_score(y_test, tuned_pred)

    baseline_acc = np.nan
    baseline_name = item["baseline_name"]
    if baseline_name is not None and isinstance(baseline_pool, dict) and baseline_name in baseline_pool:
        baseline_pred = baseline_pool[baseline_name].predict(X_test)
        baseline_acc = accuracy_score(y_test, baseline_pred)

    acc_improvement = tuned_acc - baseline_acc if not pd.isna(baseline_acc) else np.nan

    hyperparam_summary_rows.append(
        {
            "Model": item["Model"],
            "Best CV Score (macro F1)": round(getattr(grid_obj, "best_score_", np.nan), 4),
            "Hyperparameter Names": ", ".join(sorted(best_params.keys())),
            "Hyperparameter Values": "; ".join(f"{k}={v}" for k, v in sorted(best_params.items())),
            "Baseline Test Accuracy": round(baseline_acc, 4) if not pd.isna(baseline_acc) else "N/A",
            "Tuned Test Accuracy": round(tuned_acc, 4),
            "Accuracy Improvement": round(acc_improvement, 4) if not pd.isna(acc_improvement) else "N/A",
        }
    )

if hyperparam_summary_rows:
    hyperparam_summary_df = pd.DataFrame(hyperparam_summary_rows)
    display(hyperparam_summary_df)
else:
    print("No tuned model grid-search objects and predictions were found. Re-run tuning cells first.")

print("\nNotes:")
print("- Baseline Test Accuracy compares tuned models against their matching baseline model when available.")
print("- XGBoost, LightGBM, and Extra Trees are shown with full tuned hyperparameters and tuned accuracy;")
print("  they may show N/A baseline if no direct baseline counterpart was fitted in the baseline model set.")

### Sample Split and Class Distribution

This section quantifies the train/test split and class balance so the later metrics can be interpreted in context. It is especially important here because Cat 3 is the smallest class and the binary wait-time target is highly imbalanced.

Showing the sample split, the category counts, and the binary positive rate makes it clear that the model is working under uneven class distributions. That is one reason recall-based criteria are used later instead of relying on accuracy alone.

In [ ]:
print("=" * 80)
print("SAMPLE SPLIT AND CLASS DISTRIBUTION")
print("=" * 80)

# Overall split
print(f"\nOverall Data Split:")
print(f"  Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Test set: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")
print(f"  Total: {len(X)} samples")

# Training set distribution
print(f"\nTraining Set Class Distribution:")
train_counts = pd.Series(y_train).value_counts().sort_index()
for cat_num in [1, 2, 3]:
    count = train_counts.get(cat_num, 0)
    pct = (count / len(y_train)) * 100
    print(f"  Category {cat_num} (Cat {cat_num}): {count} samples ({pct:.1f}%)")

# Test set distribution
print(f"\nTest Set Class Distribution:")
test_counts = pd.Series(y_test).value_counts().sort_index()
for cat_num in [1, 2, 3]:
    count = test_counts.get(cat_num, 0)
    pct = (count / len(y_test)) * 100
    print(f"  Category {cat_num} (Cat {cat_num}): {count} samples ({pct:.1f}%)")

# Binary target (if available)
if "exceeded_wait" in df.columns:
    print(f"\n\nBinary Target (exceeded_wait):")
    print(f"Training set distribution:")
    train_binary = pd.Series(df.loc[X_train.index, "exceeded_wait"]).value_counts()
    print(f"  Did not exceed: {train_binary.get(0, 0)} samples ({train_binary.get(0, 0)/len(y_train)*100:.1f}%)")
    print(f"  Exceeded: {train_binary.get(1, 0)} samples ({train_binary.get(1, 0)/len(y_train)*100:.1f}%)")
    
    print(f"\nTest set distribution:")
    test_binary = pd.Series(df.loc[X_test.index, "exceeded_wait"]).value_counts()
    print(f"  Did not exceed: {test_binary.get(0, 0)} samples ({test_binary.get(0, 0)/len(y_test)*100:.1f}%)")
    print(f"  Exceeded: {test_binary.get(1, 0)} samples ({test_binary.get(1, 0)/len(y_test)*100:.1f}%)")

### Confusion Matrices by Model

These confusion matrices provide the most direct view of how each baseline model behaves across Cat 1, Cat 2, and Cat 3. They make it easier to see whether a model is missing urgent cases, collapsing categories together, or preserving the separation needed for a safer triage workflow.

The reports underneath the plots are included so the visual pattern and the numeric classification metrics can be read together. That combination is important because recall, precision, and F1 can tell different stories depending on which category is being examined.

In [ ]:
print("=" * 80)
print("CONFUSION MATRICES - BASELINE MODELS")
print("=" * 80)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, (model_name, model_pipeline) in enumerate(fitted_models.items()):
    y_pred = model_pipeline.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=[1, 2, 3])
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx], 
                xticklabels=["Cat 1", "Cat 2", "Cat 3"],
                yticklabels=["Cat 1", "Cat 2", "Cat 3"],
                cbar=False)
    axes[idx].set_title(f"{model_name}")
    axes[idx].set_ylabel("True Label")
    axes[idx].set_xlabel("Predicted Label")

# Hide unused subplots
for idx in range(len(fitted_models), len(axes)):
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

# Print detailed classification reports for key models
print("\n" + "=" * 80)
print("CLASSIFICATION REPORTS - BASELINE MODELS")
print("=" * 80)

for model_name in ["SVM", "Logistic Regression", "Random Forest"]:
    if model_name in fitted_models:
        y_pred = fitted_models[model_name].predict(X_test)
        print(f"\n{model_name}:")
        print(classification_report(y_test, y_pred, 
                                   target_names=["Cat 1", "Cat 2", "Cat 3"],
                                   digits=4))

### Confusion Matrices - Tuned Models

These plots compare tuned and enhanced models side by side using the same three-class target. The confusion matrices help show whether the models are improving urgent-case detection or simply trading one category of error for another.

That distinction matters here because the project is not just about maximizing overall accuracy. A model that looks good on aggregate metrics but weakens Cat 1 recall would be less useful clinically, even if it performs well on average.

In [ ]:
print("=" * 80)
print("CONFUSION MATRICES - TUNED/ENHANCED MODELS")
print("=" * 80)

tuned_models_to_plot = {}

# Collect tuned model predictions from globals
if "y_pred_log" in globals():
    tuned_models_to_plot["Tuned Logistic Regression"] = globals()["y_pred_log"]
if "y_pred_rf" in globals():
    tuned_models_to_plot["Tuned Random Forest"] = globals()["y_pred_rf"]
if "y_pred_extra" in globals():
    tuned_models_to_plot["Extra Trees"] = globals()["y_pred_extra"]
if "y_pred_xgb" in globals():
    tuned_models_to_plot["Tuned XGBoost"] = globals()["y_pred_xgb"]
if "y_pred_lgbm" in globals():
    tuned_models_to_plot["Tuned LightGBM"] = globals()["y_pred_lgbm"]

if len(tuned_models_to_plot) > 0:
    num_tuned = len(tuned_models_to_plot)
    ncols = min(3, num_tuned)
    nrows = (num_tuned + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4*nrows))
    if num_tuned == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for idx, (model_name, y_pred) in enumerate(tuned_models_to_plot.items()):
        cm = confusion_matrix(y_test, y_pred, labels=[1, 2, 3])
        
        sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", ax=axes[idx],
                    xticklabels=["Cat 1", "Cat 2", "Cat 3"],
                    yticklabels=["Cat 1", "Cat 2", "Cat 3"],
                    cbar=False)
        axes[idx].set_title(f"{model_name}")
        axes[idx].set_ylabel("True Label")
        axes[idx].set_xlabel("Predicted Label")
    
    # Hide unused subplots
    for idx in range(len(tuned_models_to_plot), len(axes)):
        axes[idx].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    # Print classification reports for tuned models
    print("\n" + "=" * 80)
    print("CLASSIFICATION REPORTS - TUNED MODELS")
    print("=" * 80)
    
    for model_name, y_pred in tuned_models_to_plot.items():
        print(f"\n{model_name}:")
        print(classification_report(y_test, y_pred,
                                   target_names=["Cat 1", "Cat 2", "Cat 3"],
                                   digits=4))
else:
    print("No tuned model predictions found.")

### Per-Category Prediction Summary

This summary breaks the SVM predictions down category by category so the model is not judged only by aggregate accuracy. That matters because Cat 1 and Cat 3 have very different safety implications.

The goal is to make it obvious where the model is strong and where it still misses cases. In particular, the Cat 3 recall and the distribution of predicted categories help show whether the model is over-predicting routine cases or holding a balanced decision boundary.

In [ ]:
print("=" * 80)
print("PER-CATEGORY PREDICTION PERFORMANCE")
print("=" * 80)

# SVM (Best model)
svm_pred = fitted_models["SVM"].predict(X_test)

print("\nSVM (Baseline Winner) - Per-Category Metrics:")
print("-" * 80)

for cat_num in [1, 2, 3]:
    y_true_binary = (y_test == cat_num).astype(int)
    y_pred_binary = (svm_pred == cat_num).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_true_binary, y_pred_binary).ravel()
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\nCategory {cat_num}:")
    print(f"  True Positives: {tp}")
    print(f"  False Positives: {fp}")
    print(f"  False Negatives: {fn}")
    print(f"  True Negatives: {tn}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")

# Summary table
print("\n" + "=" * 80)
print("CATEGORY DISTRIBUTION IN TEST PREDICTIONS (SVM)")
print("=" * 80)

true_dist = pd.Series(y_test).value_counts().sort_index()
pred_dist = pd.Series(svm_pred).value_counts().sort_index()

summary_data = []
for cat_num in [1, 2, 3]:
    summary_data.append({
        "Category": f"Cat {cat_num}",
        "Actual Count": true_dist.get(cat_num, 0),
        "Actual %": f"{true_dist.get(cat_num, 0) / len(y_test) * 100:.1f}%",
        "Predicted Count": pred_dist.get(cat_num, 0),
        "Predicted %": f"{pred_dist.get(cat_num, 0) / len(svm_pred) * 100:.1f}%",
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

### Feature Importance and Symptom Contributions

Feature importance is computed for **every model** using the most appropriate method available:

| Method | Applied to |
|--------|------------|
| **Native (Gini / gain)** | Decision Tree, Random Forest, Extra Trees, Gradient Boosting, XGBoost, LightGBM |
| **Native (\|coef\|)** | Logistic Regression, linear SVM |
| **Permutation (macro F1)** | KNN, SVM (rbf/poly kernel), Dummy — any model without a native importance attribute |

Permutation importance shuffles each feature in turn and measures how much the macro F1 drops, so it is model-agnostic and directly linked to the evaluation metric.

Clinically, features such as `ind_melaena`, `ind_fobt`, `ind_pr_bleed`, and `ind_therapeutic` are expected near the top of urgency-sensitive models. If `risk_score` or `age_yrs` dominate instead, that is worth investigating for potential leakage or confounding.


In [ ]:
print('=' * 80)
print('FEATURE IMPORTANCE — ALL MODELS')
print('=' * 80)

from sklearn.inspection import permutation_importance

TOP_N = 15

def _feat_names(pipeline_obj):
    pre = pipeline_obj.named_steps.get('preprocess')
    if pre is not None and hasattr(pre, 'get_feature_names_out'):
        try:
            return list(pre.get_feature_names_out())
        except Exception:
            pass
    return None

def _get_importance(name, pipe, X_eval, y_eval):
    if not hasattr(pipe, 'named_steps'):
        return None, None
    model = pipe.named_steps.get('model')
    fnames = _feat_names(pipe)

    # Tree-based native importance
    if model is not None and hasattr(model, 'feature_importances_'):
        imp = np.asarray(model.feature_importances_)
        method = 'native (Gini/gain)'
    # Linear model: mean |coef| across classes
    elif model is not None and hasattr(model, 'coef_'):
        coefs = np.asarray(model.coef_)
        imp = np.mean(np.abs(coefs), axis=0) if coefs.ndim > 1 else np.abs(coefs)
        method = 'native (|coef|)'
    # Universal fallback: permutation importance
    else:
        try:
            perm = permutation_importance(
                pipe, X_eval, y_eval,
                n_repeats=3, random_state=42,
                scoring='f1_macro', n_jobs=-1,
            )
            imp = perm.importances_mean
            fnames = list(X_eval.columns)
            method = 'permutation (macro F1)'
        except Exception as exc:
            print(f'  SKIPPED {name}: {exc}')
            return None, None

    if fnames is None or len(fnames) != len(imp):
        fnames = [f'f{i}' for i in range(len(imp))]

    df = (
        pd.DataFrame({'feature': fnames, 'importance': imp})
        .sort_values('importance', ascending=False)
        .reset_index(drop=True)
    )
    return df, method


# Build registry ─────────────────────────────────────────────────────────
def _from_fitted(key):
    fm = globals().get('fitted_models', {})
    return fm.get(key) if isinstance(fm, dict) else None

model_registry = [
    ('Baseline — Dummy',               _from_fitted('Dummy')),
    ('Baseline — Logistic Regression',  _from_fitted('Logistic Regression')),
    ('Baseline — Decision Tree',        _from_fitted('Decision Tree')),
    ('Baseline — Random Forest',        _from_fitted('Random Forest')),
    ('Baseline — Gradient Boosting',    _from_fitted('Gradient Boosting')),
    ('Baseline — SVM',                  _from_fitted('SVM')),
    ('Baseline — KNN',                  _from_fitted('KNN')),
    ('Tuned — Logistic Regression',     globals().get('best_log_reg')),
    ('Tuned — Extra Trees (fixed)',      globals().get('extra_trees_pipeline')),
    ('Tuned — Extra Trees',             globals().get('best_et')),
    ('Tuned — Decision Tree',           globals().get('best_dt')),
    ('Tuned — Gradient Boosting',       globals().get('best_gb')),
    ('Tuned — Random Forest',           globals().get('best_rf')),
    ('Tuned — KNN',                     globals().get('best_knn')),
    ('Tuned — SVM',                     globals().get('best_svm')),
    ('Tuned — XGBoost',                 globals().get('best_xgb')),
    ('Tuned — LightGBM',                globals().get('best_lgbm')),
]

active = [(n, m) for n, m in model_registry
          if m is not None and hasattr(m, 'predict')]
print(f'Computing feature importance for {len(active)} models …')

results = {}
for name, pipe in active:
    imp_df, method = _get_importance(name, pipe, X_test, y_test)
    if imp_df is not None:
        results[name] = (imp_df, method)


# Plot bar-chart grid ─────────────────────────────────────────────────────
palette = {'native (Gini/gain)': 'steelblue',
           'native (|coef|)':    'seagreen',
           'permutation (macro F1)': 'coral'}

ncols = 3
nrows = -(-len(results) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 3.5))
axes = axes.flatten()

for ax_i, (name, (imp_df, method)) in enumerate(results.items()):
    top = imp_df.head(TOP_N)
    color = palette.get(method, 'steelblue')
    ax = axes[ax_i]
    ax.barh(top['feature'][::-1], top['importance'][::-1], color=color)
    ax.set_title(f'{name}\n({method})', fontsize=8)
    ax.set_xlabel('Importance', fontsize=7)
    ax.tick_params(axis='y', labelsize=6)

for j in range(ax_i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    f'Feature Importance — Top {TOP_N} Features per Model\n'
    'Blue = Gini/gain  |  Green = |coef|  |  Coral = permutation',
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()


# Summary table ──────────────────────────────────────────────────────────
print('\nTOP FEATURE PER MODEL')
summary = [
    {
        'Model': n,
        'Importance Method': method,
        'Top Feature': df.iloc[0]['feature'],
        'Importance Score': round(float(df.iloc[0]['importance']), 4),
    }
    for n, (df, method) in results.items()
]
display(pd.DataFrame(summary))


### Hyperparameter Tuning Details

This table is intended to make the tuning process transparent. It records the best hyperparameters found for each tuned model so the final comparison is reproducible and easy to audit.

The key point is that the tuned models were genuinely optimized, while SVM is intentionally left as a baseline configuration. That contrast matters because the final ranking is more meaningful if it can show whether a default model or a tuned model is actually the safest choice.

In [ ]:
print("=" * 80)
print("HYPERPARAMETER TUNING DETAILS BY MODEL")
print("=" * 80)

# Grid search results from globals
grid_search_results = []

# Logistic Regression
if "log_best_params" in globals():
    print("\nLogistic Regression - Best Hyperparameters:")
    best_params = globals()["log_best_params"]
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    if "log_best_score" in globals():
        print(f"  Best cross-validation score: {globals()['log_best_score']:.4f}")

# Random Forest
if "rf_best_params" in globals():
    print("\nRandom Forest - Best Hyperparameters:")
    best_params = globals()["rf_best_params"]
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    if "rf_best_score" in globals():
        print(f"  Best cross-validation score: {globals()['rf_best_score']:.4f}")

# Extra Trees
if "extra_best_params" in globals():
    print("\nExtra Trees - Best Hyperparameters:")
    best_params = globals()["extra_best_params"]
    for key, value in best_params.items():
        print(f"  {key}: {value}")

# XGBoost
if "xgb_best_params" in globals():
    print("\nXGBoost - Best Hyperparameters:")
    best_params = globals()["xgb_best_params"]
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    if "xgb_best_score" in globals():
        print(f"  Best cross-validation score: {globals()['xgb_best_score']:.4f}")

# LightGBM
if "lgbm_best_params" in globals():
    print("\nLightGBM - Best Hyperparameters:")
    best_params = globals()["lgbm_best_params"]
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    if "lgbm_best_score" in globals():
        print(f"  Best cross-validation score: {globals()['lgbm_best_score']:.4f}")

# SVM (baseline, no tuning)
print("\nSVM (Baseline - No Grid Search):")
print(f"  kernel: rbf")
print(f"  C: 2.0")
print(f"  gamma: scale")
print(f"  class_weight: balanced")
print(f"  probability: True")
print(f"  Uses scikit-learn default configuration (default params only)")

### Binary Classification Analysis

This binary task is a useful side check because it measures whether the same feature set can identify patients who exceeded their recommended wait time. The class imbalance is strong, so the main point here is not raw accuracy but whether the model can recover the minority positive class at all.

The confusion matrix and classification report below show how difficult this target is. That supports the broader conclusion that missed urgent cases are the central risk, and it is why Cat 1 recall remains the main evaluation priority throughout the notebook.

In [ ]:
if "exceeded_wait" in df.columns:
    print("=" * 80)
    print("BINARY TARGET ANALYSIS: WAIT TIME EXCEEDED")
    print("=" * 80)
    
    y_binary_train = df.loc[X_train.index, "exceeded_wait"]
    y_binary_test = df.loc[X_test.index, "exceeded_wait"]
    
    print("\nBinary Target Distribution:")
    print(f"  Class 0 (Did Not Exceed): {(y_binary_test == 0).sum()} ({(y_binary_test == 0).sum()/len(y_binary_test)*100:.1f}%)")
    print(f"  Class 1 (Exceeded): {(y_binary_test == 1).sum()} ({(y_binary_test == 1).sum()/len(y_binary_test)*100:.1f}%)")
    
    # Logistic Regression for binary classification
    from sklearn.linear_model import LogisticRegression as LR
    
    binary_lr = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", LR(max_iter=2000, class_weight="balanced", random_state=42))
    ])
    
    binary_lr.fit(X_train, y_binary_train)
    y_binary_pred = binary_lr.predict(X_test)
    
    print("\nBinary Classification (Logistic Regression):")
    print(f"  Accuracy: {accuracy_score(y_binary_test, y_binary_pred):.4f}")
    print(f"  Precision: {precision_score(y_binary_test, y_binary_pred):.4f}")
    print(f"  Recall: {recall_score(y_binary_test, y_binary_pred):.4f}")
    print(f"  F1-Score: {f1_score(y_binary_test, y_binary_pred):.4f}")
    
    print("\nBinary Classification Report:")
    print(classification_report(y_binary_test, y_binary_pred, 
                               target_names=["Did Not Exceed", "Exceeded"],
                               digits=4))
    
    # Confusion matrix
    cm_binary = confusion_matrix(y_binary_test, y_binary_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_binary, annot=True, fmt="d", cmap="RdYlGn_r", ax=ax,
                xticklabels=["Did Not Exceed", "Exceeded"],
                yticklabels=["Did Not Exceed", "Exceeded"])
    ax.set_title("Binary Classification - Wait Time Exceeded")
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()
else:
    print("=" * 80)
    print("NO BINARY TARGET AVAILABLE")
    print("=" * 80)
    print("The 'exceeded_wait' column was not found in the dataset.")

### Bayesian Network Analysis

This section tests whether symptom indicators improve referral-category prediction in a probabilistic setting. To isolate the effect of symptoms, the Bayesian model is evaluated twice: once with all available pre-procedure features, and once with the symptom indicators removed.

The comparison is useful because it answers a specific clinical question: do symptom signals add measurable value beyond demographics, referral metadata, and other pre-procedure variables? The results below show that they do, especially for Cat 1 recall, which is the most safety-critical metric in this project.

In [ ]:
print("=" * 80)
print("BAYESIAN NETWORK ANALYSIS: SYMPTOM EFFECT ON PREDICTION")
print("=" * 80)

from sklearn.naive_bayes import BernoulliNB
from sklearn.preprocessing import FunctionTransformer, KBinsDiscretizer, OneHotEncoder

symptom_cols = [col for col in X.columns if col.startswith("ind_")]
non_symptom_cols = [col for col in X.columns if col not in symptom_cols]

print(f"\nSymptom indicator columns found: {len(symptom_cols)}")
print(f"Other pre-procedure columns retained in the network: {len(non_symptom_cols)}")


def build_bayesian_pipeline(selected_columns):
    selected_frame = X[selected_columns]

    binary_cols = [
        col
        for col in selected_columns
        if col not in categorical_cols and selected_frame[col].nunique(dropna=True) <= 2
    ]
    categorical_like_cols = [col for col in selected_columns if col in categorical_cols]
    continuous_cols_local = [
        col
        for col in selected_columns
        if col not in binary_cols and col not in categorical_like_cols
    ]

    transformers = []

    if binary_cols:
        transformers.append(
            (
                "binary",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("to_int", FunctionTransformer(lambda values: values.astype(int), validate=False, feature_names_out="one-to-one")),
                    ]
                ),
                binary_cols,
            )
        )

    if continuous_cols_local:
        transformers.append(
            (
                "continuous",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("bins", KBinsDiscretizer(n_bins=4, encode="onehot-dense", strategy="quantile")),
                    ]
                ),
                continuous_cols_local,
            )
        )

    if categorical_like_cols:
        transformers.append(
            (
                "categorical",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical_like_cols,
            )
        )

    preprocessor_bn = ColumnTransformer(transformers=transformers, remainder="drop")
    bayes_model = Pipeline(
        steps=[
            ("preprocess", preprocessor_bn),
            ("model", BernoulliNB(alpha=1.0)),
        ]
    )
    return bayes_model


def evaluate_bayesian_model(model_name, selected_columns):
    bayes_pipeline = build_bayesian_pipeline(selected_columns)
    bayes_pipeline.fit(X_train[selected_columns], y_train)
    y_pred_bayes = bayes_pipeline.predict(X_test[selected_columns])

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred_bayes),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred_bayes),
        "Macro F1": f1_score(y_test, y_pred_bayes, average="macro", zero_division=0),
        "Cat 1 Recall": recall_score((y_test == 1).astype(int), (y_pred_bayes == 1).astype(int), zero_division=0),
        "Cat 2 Recall": recall_score((y_test == 2).astype(int), (y_pred_bayes == 2).astype(int), zero_division=0),
        "Cat 3 Recall": recall_score((y_test == 3).astype(int), (y_pred_bayes == 3).astype(int), zero_division=0),
    }
    return bayes_pipeline, y_pred_bayes, metrics


with_symptoms_pipeline, y_pred_with_symptoms, with_symptoms_metrics = evaluate_bayesian_model(
    "Bayesian Network (with symptoms)",
    list(X.columns),
)

without_symptoms_pipeline, y_pred_without_symptoms, without_symptoms_metrics = evaluate_bayesian_model(
    "Bayesian Network (without symptoms)",
    non_symptom_cols,
)

bayes_results_df = pd.DataFrame([with_symptoms_metrics, without_symptoms_metrics])
display(bayes_results_df.round(4))

print("\nMetric change when symptom indicators are included:")
comparison_columns = ["Accuracy", "Balanced Accuracy", "Macro F1", "Cat 1 Recall", "Cat 2 Recall", "Cat 3 Recall"]
for metric_name in comparison_columns:
    delta = with_symptoms_metrics[metric_name] - without_symptoms_metrics[metric_name]
    sign = "+" if delta >= 0 else ""
    print(f"  {metric_name}: {sign}{delta:.4f}")

print("\nConfusion matrices for the Bayesian models:")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for axis, title, predictions in [
    (axes[0], "With Symptoms", y_pred_with_symptoms),
    (axes[1], "Without Symptoms", y_pred_without_symptoms),
]:
    cm = confusion_matrix(y_test, predictions, labels=[1, 2, 3])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Purples",
        ax=axis,
        xticklabels=["Cat 1", "Cat 2", "Cat 3"],
        yticklabels=["Cat 1", "Cat 2", "Cat 3"],
        cbar=False,
    )
    axis.set_title(title)
    axis.set_xlabel("Predicted Label")
    axis.set_ylabel("True Label")

plt.tight_layout()
plt.show()

print("\nSymptom contribution summary from the Bayesian network:")
preprocessed_names = with_symptoms_pipeline.named_steps["preprocess"].get_feature_names_out()
raw_feature_names = np.array([name.split("__", 1)[-1] for name in preprocessed_names])
symptom_feature_mask = np.array([name.startswith("ind_") for name in raw_feature_names])

if symptom_feature_mask.any():
    symptom_feature_names = raw_feature_names[symptom_feature_mask]
    training_symptom_frame = X_train[symptom_feature_names].apply(pd.to_numeric, errors="coerce").fillna(0)
    training_symptom_prevalence = (training_symptom_frame > 0).mean()
    bayes_estimator = with_symptoms_pipeline.named_steps["model"]

    for class_index, class_label in enumerate(bayes_estimator.classes_):
        class_probs = np.exp(bayes_estimator.feature_log_prob_[class_index][symptom_feature_mask])
        symptom_effect_df = pd.DataFrame(
            {
                "Symptom": symptom_feature_names,
                "P(symptom=1 | class)": class_probs,
                "Training prevalence": training_symptom_prevalence.loc[symptom_feature_names].values,
            }
        )
        symptom_effect_df["Lift"] = symptom_effect_df["P(symptom=1 | class)"] / (symptom_effect_df["Training prevalence"] + 1e-9)
        symptom_effect_df = symptom_effect_df.sort_values("Lift", ascending=False)

        print(f"\nTop symptom signals for Cat {class_label}:")
        display(symptom_effect_df.head(10).round(4))
else:
    print("No symptom indicator columns were detected for Bayesian analysis.")

### Bayesian Recall Summary

In [ ]:
print("=" * 80)
print("BAYESIAN RECALL SUMMARY")
print("=" * 80)

summary_rows = []
for model_name, metrics in [
    ("With Symptoms", with_symptoms_metrics),
    ("Without Symptoms", without_symptoms_metrics),
]:
    summary_rows.append({
        "Model": model_name,
        "Cat 1 Recall": metrics["Cat 1 Recall"],
        "Cat 2 Recall": metrics["Cat 2 Recall"],
        "Cat 3 Recall": metrics["Cat 3 Recall"],
        "Balanced Accuracy": metrics["Balanced Accuracy"],
        "Macro F1": metrics["Macro F1"],
    })

bayes_recall_summary_df = pd.DataFrame(summary_rows)
display(bayes_recall_summary_df.round(4))

print("\nRecall gain from including symptom indicators:")
print(f"  Cat 1 Recall: {with_symptoms_metrics['Cat 1 Recall'] - without_symptoms_metrics['Cat 1 Recall']:+.4f}")
print(f"  Cat 2 Recall: {with_symptoms_metrics['Cat 2 Recall'] - without_symptoms_metrics['Cat 2 Recall']:+.4f}")
print(f"  Cat 3 Recall: {with_symptoms_metrics['Cat 3 Recall'] - without_symptoms_metrics['Cat 3 Recall']:+.4f}")

## Model Exploration

### Baseline Models
Baseline models (Dummy, Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, SVM, KNN) were trained first using default scikit-learn hyperparameters. **SVM emerged as the best baseline model** with strong recall across all categories (Cat 1: 97.27%, Cat 2: 96.88%, Cat 3: 64.71%) and 94.62% test accuracy.

### Tuned Models
Tuned models were trained using grid search to find optimal hyperparameters:
- **Logistic Regression:** Grid-searched on C and solver
- **Random Forest:** Grid-searched on n_estimators, max_depth, min_samples_leaf, min_samples_split
- **Extra Trees:** Configured with enhanced hyperparameters
- **XGBoost:** Grid-searched on n_estimators, max_depth, learning_rate, subsample, colsample_bytree
- **LightGBM:** Grid-searched on n_estimators, max_depth, learning_rate, num_leaves, min_child_samples

Among tuned models, **LightGBM ranked first** with 94.07% accuracy and 87.27% balanced accuracy, while maintaining Cat 1 recall of 97.27%.

### Model Improvement Experiment
A soft-voting ensemble built from the strongest fitted models improved the recall-weighted score slightly while keeping Cat 1 recall unchanged. That makes it the best overall candidate under the unified ranking criteria, even though SVM remains the strongest single model.

### Unified Ranking and Final Selection
However, under the unified ranking criteria (macro F2 → Cat 1 recall → macro F1), which emphasizes recall safety for urgent cases, the **soft-voting ensemble now ranks first overall** when compared against all baseline, tuned, and ensemble candidates.

This is significant because:
- SVM still achieved the strongest single-model baseline performance using default hyperparameters only
- The ensemble combines complementary decision boundaries from the best fitted models
- The improvement is small but consistent across macro F2 and macro F1, with Cat 1 recall preserved

### Conclusion
**The soft-voting ensemble is the final selected model** for this notebook because it provides the best recall-weighted balance under the unified ranking criteria. SVM remains the best individual model, but the ensemble is slightly stronger overall while keeping urgent-case recall unchanged.

### Detailed Classification Analysis

This section ties the notebook together by summarizing the core findings across modelling, tuning, interpretability, and class balance. It should give the reader a clear path from the data split through model comparison to the final recommendation.

The detailed classification analysis is especially useful because it captures the practical implications of the model choices: how often urgent cases are found, how symptom signals shift predictions, and whether the binary wait-time problem is learnable from the available features.

### Model Improvement Experiment

This section pulls the improvement attempts together and shows the progression from the strongest single model to the ensemble variants. The main finding is that SVM remains the strongest individual baseline, but the ensemble-based approaches edge it out slightly when the ranking prioritises macro F2 and urgent-case recall.

That outcome is important because it confirms the notebook is not relying on one lucky model choice. The improvement experiments show that the final recommendation is stable across multiple checks, and that symptom-aware or weighted variants do not overturn the conclusion that a soft-voting ensemble is the safest overall choice.

In [ ]:
print("=" * 80)
print("MODEL IMPROVEMENT EXPERIMENT: SOFT-VOTING ENSEMBLE")
print("=" * 80)

from sklearn.ensemble import VotingClassifier

ensemble_estimators = []

candidate_estimators = {
    "SVM": globals().get("fitted_models", {}).get("SVM"),
    "Logistic Regression": globals().get("best_log_reg", None),
    "Random Forest": globals().get("best_rf", None),
    "Extra Trees": globals().get("extra_trees_pipeline", None),
    "XGBoost": globals().get("best_xgb", None),
    "LightGBM": globals().get("best_lgbm", None),
}

for model_name, estimator in candidate_estimators.items():
    if estimator is not None:
        ensemble_estimators.append((model_name.replace(" ", "_"), estimator))

if len(ensemble_estimators) >= 3:
    ensemble_model = VotingClassifier(
        estimators=ensemble_estimators,
        voting="soft",
        n_jobs=-1,
        weights=[1] * len(ensemble_estimators),
    )
    ensemble_model.fit(X_train, y_train)
    ensemble_predictions = ensemble_model.predict(X_test)

    ensemble_metrics = {
        "Accuracy": accuracy_score(y_test, ensemble_predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_test, ensemble_predictions),
        "Macro Precision": precision_score(y_test, ensemble_predictions, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_test, ensemble_predictions, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, ensemble_predictions, average="macro", zero_division=0),
        "Macro F2": fbeta_score(y_test, ensemble_predictions, beta=2, average="macro", zero_division=0),
        "Cat 1 Recall": recall_score((y_test == 1).astype(int), (ensemble_predictions == 1).astype(int), zero_division=0),
        "Cat 2 Recall": recall_score((y_test == 2).astype(int), (ensemble_predictions == 2).astype(int), zero_division=0),
        "Cat 3 Recall": recall_score((y_test == 3).astype(int), (ensemble_predictions == 3).astype(int), zero_division=0),
    }

    if "all_results_df" in globals() and len(all_results_df) > 0:
        comparison_snapshot = all_results_df[["model", "macro_f2", "cat1_recall", "macro_f1"]].head(5).copy()
        display(comparison_snapshot)

    print("\nSoft-voting ensemble metrics:")
    for metric_name, value in ensemble_metrics.items():
        print(f"  {metric_name}: {value:.4f}")

    if "all_results_df" in globals() and len(all_results_df) > 0:
        current_best = all_results_df.iloc[0]
        print("\nBest existing model:")
        print(f"  {current_best['model']}")
        print(f"  Macro F2: {current_best['macro_f2']:.4f}")
        print(f"  Cat 1 Recall: {current_best['cat1_recall']:.4f}")
        print(f"  Macro F1: {current_best['macro_f1']:.4f}")
        print("\nDifference versus current best:")
        print(f"  Macro F2: {ensemble_metrics['Macro F2'] - current_best['macro_f2']:+.4f}")
        print(f"  Cat 1 Recall: {ensemble_metrics['Cat 1 Recall'] - current_best['cat1_recall']:+.4f}")
        print(f"  Macro F1: {ensemble_metrics['Macro F1'] - current_best['macro_f1']:+.4f}")

    print("\nClassification report for ensemble:")
    print(
        classification_report(
            y_test,
            ensemble_predictions,
            labels=LABELS,
            target_names=TARGET_NAMES,
            zero_division=0,
        )
    )

    cm = confusion_matrix(y_test, ensemble_predictions, labels=LABELS)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Oranges",
        xticklabels=TARGET_NAMES,
        yticklabels=TARGET_NAMES,
        ax=ax,
    )
    ax.set_title("Confusion Matrix: Soft-Voting Ensemble")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.show()

    if "all_results_df" in globals() and len(all_results_df) > 0:
        if ensemble_metrics["Macro F2"] > float(all_results_df.iloc[0]["macro_f2"]):
            print("\nThis ensemble improves the current best macro F2 score.")
        else:
            print("\nThis ensemble does not outperform the current best model on macro F2, but it remains a useful safety check.")
else:
    print("Not enough fitted base estimators were available to build a soft-voting ensemble.")

### Weighted Ensemble Search

The weighted search tested whether giving extra influence to one model, especially SVM or the stronger tree-based learners, could improve the recall-focused objective. This was a useful follow-up because the equal-weight ensemble already performed well, so the question was whether the model combination could be made more clinically selective.

The search showed only a small change in the ranking metrics, and the best weight configuration did not materially outperform the equal-weight version. In practical terms, that means the ensemble was already close to stable, and changing the voting weights mostly shifted the balance without creating a better urgent-case detector.

In [ ]:
print("=" * 80)
print("MODEL IMPROVEMENT EXPERIMENT: WEIGHTED SOFT-VOTING SEARCH")
print("=" * 80)

from sklearn.ensemble import VotingClassifier

weighted_candidates = {
    "SVM": globals().get("fitted_models", {}).get("SVM"),
    "Random Forest": globals().get("best_rf", None),
    "LightGBM": globals().get("best_lgbm", None),
    "XGBoost": globals().get("best_xgb", None),
}

available_estimators = [(name.replace(" ", "_"), model) for name, model in weighted_candidates.items() if model is not None]

if len(available_estimators) >= 3:
    search_space = [
        (1, 1, 1, 1),
        (2, 1, 1, 1),
        (1, 2, 1, 1),
        (1, 1, 2, 1),
        (1, 1, 1, 2),
        (3, 1, 1, 1),
        (1, 3, 1, 1),
        (1, 1, 3, 1),
        (1, 1, 1, 3),
        (2, 2, 1, 1),
        (2, 1, 2, 1),
        (2, 1, 1, 2),
        (1, 2, 2, 1),
        (1, 2, 1, 2),
        (1, 1, 2, 2),
        (3, 2, 1, 1),
        (3, 1, 2, 1),
        (3, 1, 1, 2),
    ]

    inner_train_X, inner_val_X, inner_train_y, inner_val_y = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        stratify=y_train,
        random_state=RANDOM_STATE,
    )

    weight_search_rows = []
    best_weighted_model = None
    best_weighted_metrics = None
    best_weighted_weights = None

    for weights in search_space:
        candidate = VotingClassifier(
            estimators=available_estimators,
            voting="soft",
            weights=list(weights),
            n_jobs=-1,
        )
        candidate.fit(inner_train_X, inner_train_y)
        val_predictions = candidate.predict(inner_val_X)
        row = {
            "weights": weights,
            "macro_f2": fbeta_score(inner_val_y, val_predictions, beta=2, average="macro", zero_division=0),
            "cat1_recall": recall_score((inner_val_y == 1).astype(int), (val_predictions == 1).astype(int), zero_division=0),
            "macro_f1": f1_score(inner_val_y, val_predictions, average="macro", zero_division=0),
        }
        weight_search_rows.append(row)

    weight_search_df = pd.DataFrame(weight_search_rows).sort_values(
        by=["macro_f2", "cat1_recall", "macro_f1"],
        ascending=False,
    ).reset_index(drop=True)
    display(weight_search_df)

    best_weighted_weights = weight_search_df.iloc[0]["weights"]
    print(f"\nBest validation weights: {best_weighted_weights}")

    best_weighted_model = VotingClassifier(
        estimators=available_estimators,
        voting="soft",
        weights=list(best_weighted_weights),
        n_jobs=-1,
    )
    best_weighted_model.fit(X_train, y_train)
    weighted_test_predictions = best_weighted_model.predict(X_test)

    best_weighted_metrics = {
        "Accuracy": accuracy_score(y_test, weighted_test_predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_test, weighted_test_predictions),
        "Macro Precision": precision_score(y_test, weighted_test_predictions, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_test, weighted_test_predictions, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, weighted_test_predictions, average="macro", zero_division=0),
        "Macro F2": fbeta_score(y_test, weighted_test_predictions, beta=2, average="macro", zero_division=0),
        "Cat 1 Recall": recall_score((y_test == 1).astype(int), (weighted_test_predictions == 1).astype(int), zero_division=0),
        "Cat 2 Recall": recall_score((y_test == 2).astype(int), (weighted_test_predictions == 2).astype(int), zero_division=0),
        "Cat 3 Recall": recall_score((y_test == 3).astype(int), (weighted_test_predictions == 3).astype(int), zero_division=0),
    }

    print("\nWeighted ensemble test metrics:")
    for metric_name, value in best_weighted_metrics.items():
        print(f"  {metric_name}: {value:.4f}")

    if "all_results_df" in globals() and len(all_results_df) > 0:
        current_best = all_results_df.iloc[0]
        print("\nDifference versus current best model:")
        print(f"  Macro F2: {best_weighted_metrics['Macro F2'] - float(current_best['macro_f2']):+.4f}")
        print(f"  Cat 1 Recall: {best_weighted_metrics['Cat 1 Recall'] - float(current_best['cat1_recall']):+.4f}")
        print(f"  Macro F1: {best_weighted_metrics['Macro F1'] - float(current_best['macro_f1']):+.4f}")

    print("\nClassification report for weighted ensemble:")
    print(
        classification_report(
            y_test,
            weighted_test_predictions,
            labels=LABELS,
            target_names=TARGET_NAMES,
            zero_division=0,
        )
    )

    cm = confusion_matrix(y_test, weighted_test_predictions, labels=LABELS)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Greens",
        xticklabels=TARGET_NAMES,
        yticklabels=TARGET_NAMES,
        ax=ax,
    )
    ax.set_title("Confusion Matrix: Weighted Soft-Voting Ensemble")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.show()

    if best_weighted_metrics["Macro F2"] > ensemble_metrics["Macro F2"]:
        print("\nThe weighted ensemble improves on the equal-weight soft-voting ensemble.")
    else:
        print("\nThe weighted ensemble did not beat the equal-weight ensemble, but it provides a calibrated improvement search.")
else:
    print("Not enough fitted estimators were available to run the weighted ensemble search.")

### Symptom-Augmented Ensemble

This experiment added symptom burden features to the ensemble explicitly, including a total symptom count and grouped symptom scores for bleeding, surveillance-related, and urgency-related indications. The aim was to test whether symptom structure, rather than only raw symptom flags, could give the model a stronger clinical signal.

The result was directionally sensible but not better than the simpler equal-weight ensemble. That suggests the model was already capturing most of the usable symptom information through the existing pre-procedure features, and the added symptom aggregates did not produce a meaningful gain on the recall-weighted ranking.

In [ ]:
print("=" * 80)
print("MODEL IMPROVEMENT EXPERIMENT: SYMPTOM-AUGMENTED ENSEMBLE")
print("=" * 80)

from sklearn.ensemble import VotingClassifier

symptom_columns = [col for col in X.columns if col.startswith("ind_")]

if len(symptom_columns) > 0:
    X_train_symptom = X_train.copy()
    X_test_symptom = X_test.copy()

    X_train_symptom["symptom_count"] = X_train_symptom[symptom_columns].sum(axis=1)
    X_test_symptom["symptom_count"] = X_test_symptom[symptom_columns].sum(axis=1)

    X_train_symptom["symptom_any"] = (X_train_symptom["symptom_count"] > 0).astype(int)
    X_test_symptom["symptom_any"] = (X_test_symptom["symptom_count"] > 0).astype(int)

    bleeding_symptoms = [col for col in ["ind_pr_bleed", "ind_melaena", "ind_anaemia"] if col in symptom_columns]
    surveillance_symptoms = [col for col in ["ind_polyp_surveillance", "ind_ca_surveillance", "ind_family_hx", "ind_ibd"] if col in symptom_columns]
    urgency_symptoms = [col for col in ["ind_therapeutic", "ind_fobt", "ind_melaena", "ind_pr_bleed"] if col in symptom_columns]

    X_train_symptom["bleeding_symptom_score"] = X_train_symptom[bleeding_symptoms].sum(axis=1) if len(bleeding_symptoms) > 0 else 0
    X_test_symptom["bleeding_symptom_score"] = X_test_symptom[bleeding_symptoms].sum(axis=1) if len(bleeding_symptoms) > 0 else 0

    X_train_symptom["surveillance_symptom_score"] = X_train_symptom[surveillance_symptoms].sum(axis=1) if len(surveillance_symptoms) > 0 else 0
    X_test_symptom["surveillance_symptom_score"] = X_test_symptom[surveillance_symptoms].sum(axis=1) if len(surveillance_symptoms) > 0 else 0

    X_train_symptom["urgency_symptom_score"] = X_train_symptom[urgency_symptoms].sum(axis=1) if len(urgency_symptoms) > 0 else 0
    X_test_symptom["urgency_symptom_score"] = X_test_symptom[urgency_symptoms].sum(axis=1) if len(urgency_symptoms) > 0 else 0

    augmented_numeric_cols = X_train_symptom.select_dtypes(include=[np.number]).columns.tolist()
    augmented_categorical_cols = [col for col in categorical_cols if col in X_train_symptom.columns]
    augmented_preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                augmented_numeric_cols,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                augmented_categorical_cols,
            ),
        ],
        remainder="drop",
    )

    augmented_estimators = []
    augmented_candidate_estimators = {
        "SVM": SVC(kernel="rbf", C=2.0, gamma="scale", class_weight="balanced", probability=True, random_state=RANDOM_STATE),
        "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", multi_class="multinomial", solver="lbfgs", random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1),
        "LightGBM": globals().get("lgbm_best_model", None).named_steps["model"] if globals().get("lgbm_best_model", None) is not None else None,
    }

    for model_name, estimator in augmented_candidate_estimators.items():
        if estimator is not None:
            augmented_estimators.append((model_name.replace(" ", "_"), estimator))

    if len(augmented_estimators) >= 3:
        symptom_augmented_ensemble = VotingClassifier(
            estimators=[
                (name, Pipeline(steps=[("preprocess", augmented_preprocessor), ("model", estimator)]))
                for name, estimator in augmented_estimators
            ],
            voting="soft",
            n_jobs=-1,
            weights=[1] * len(augmented_estimators),
        )
        symptom_augmented_ensemble.fit(X_train_symptom, y_train)
        symptom_augmented_predictions = symptom_augmented_ensemble.predict(X_test_symptom)

        symptom_augmented_metrics = {
            "Accuracy": accuracy_score(y_test, symptom_augmented_predictions),
            "Balanced Accuracy": balanced_accuracy_score(y_test, symptom_augmented_predictions),
            "Macro Precision": precision_score(y_test, symptom_augmented_predictions, average="macro", zero_division=0),
            "Macro Recall": recall_score(y_test, symptom_augmented_predictions, average="macro", zero_division=0),
            "Macro F1": f1_score(y_test, symptom_augmented_predictions, average="macro", zero_division=0),
            "Macro F2": fbeta_score(y_test, symptom_augmented_predictions, beta=2, average="macro", zero_division=0),
            "Cat 1 Recall": recall_score((y_test == 1).astype(int), (symptom_augmented_predictions == 1).astype(int), zero_division=0),
            "Cat 2 Recall": recall_score((y_test == 2).astype(int), (symptom_augmented_predictions == 2).astype(int), zero_division=0),
            "Cat 3 Recall": recall_score((y_test == 3).astype(int), (symptom_augmented_predictions == 3).astype(int), zero_division=0),
        }

        print("\nSymptom-augmented ensemble metrics:")
        for metric_name, value in symptom_augmented_metrics.items():
            print(f"  {metric_name}: {value:.4f}")

        if "all_results_df" in globals() and len(all_results_df) > 0:
            current_best = all_results_df.iloc[0]
            print("\nDifference versus current best model:")
            print(f"  Macro F2: {symptom_augmented_metrics['Macro F2'] - float(current_best['macro_f2']):+.4f}")
            print(f"  Cat 1 Recall: {symptom_augmented_metrics['Cat 1 Recall'] - float(current_best['cat1_recall']):+.4f}")
            print(f"  Macro F1: {symptom_augmented_metrics['Macro F1'] - float(current_best['macro_f1']):+.4f}")

        print("\nClassification report for symptom-augmented ensemble:")
        print(
            classification_report(
                y_test,
                symptom_augmented_predictions,
                labels=LABELS,
                target_names=TARGET_NAMES,
                zero_division=0,
            )
        )

        cm = confusion_matrix(y_test, symptom_augmented_predictions, labels=LABELS)
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Purples",
            xticklabels=TARGET_NAMES,
            yticklabels=TARGET_NAMES,
            ax=ax,
        )
        ax.set_title("Confusion Matrix: Symptom-Augmented Ensemble")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        plt.tight_layout()
        plt.show()

        if "ensemble_metrics" in globals() and symptom_augmented_metrics["Macro F2"] > ensemble_metrics["Macro F2"]:
            print("\nThe symptom-augmented ensemble improves on the earlier equal-weight ensemble.")
        elif "ensemble_metrics" in globals():
            print("\nThe symptom-augmented ensemble did not outperform the earlier equal-weight ensemble, but it explicitly incorporates symptom burden features.")
    else:
        print("Not enough fitted estimators were available to build the symptom-augmented ensemble.")
else:
    print("No symptom indicator columns were detected in X.")

## Executive Summary

The final selected model for this run is the **soft-voting ensemble**, because it ranked first in the recall-weighted comparison using macro F2, then Cat 1 recall, then macro F1.

SVM remains the best single model, and the symptom-augmented ensemble was also tested to incorporate symptom burden explicitly, but it did not outperform the equal-weight ensemble. The result still matches the project goal of minimizing missed urgent cases while maintaining strong multiclass classification performance.